In [ ]:
### This script is used to process the 2p data and corresponding treadmill behavior data and save the processed data -- before SMI calculation
### JSY, 04/2025

### This script is used to analyze the 2p data and corresponding treadmill behavior data
### 1. Load the 2p data and treadmill behavior data
### 2. Align the 2p data and treadmill behavior data
### 3a. Find temporal offset, which yields the best alignment between 2p and behavior data
### 3b. Smooth the deconvolved traces using a 250 ms Gaussian window
### 3c. Remove inactive data points
### 4. Spatial discretization (divide the VR corridor into ~110 bins, each representing 1cm and assign each data point to its corresponding spatial bin)
### 5. Test for reliability for individual cells (calculate Pearson CC or cohen’s D)
### 6. Response Plot - plotting activity of all responsive cells (cross validation – split trials in half)
### 7. Calculate Spatial Modulation Index (SMI)


In [ ]:
# Reloads all imported modules when they change
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import datetime
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.ndimage import gaussian_filter1d

%cd "C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation"
from helper import TwoP, read_xml, time2float
from helper import SpikeSmoothing, BehavioralDataFiltering as DF, spatial_discretization as SD, ReliabilityTesting as RT, ResponseVisualization as RV, SpatialModulationIndex as SMI
from helper.SpatialModulationIndexLayerSpecific import SpatialModulationIndexLayerSpecific as SMI_Layer
from helper.detrendAdaptation import detrendAdaptation as DA

from matplotlib import rcParams
rcParams['legend.fontsize'] = 14
rcParams['axes.labelsize'] = 14
rcParams['axes.titlesize'] = 20
rcParams['xtick.labelsize'] = 14
rcParams['ytick.labelsize'] = 14

**1. Load the 2p data and treadmill behavior data**

In [ ]:
# # File path to VRlog.txt and 2p data (JSY042 and JSY038 - V1 window animals)

# # JSY042, miniwalls without landmarks, L2/3, day 5
# filepath = r'F:\2P\spmod\250614_JSY_JSY042_SpatialModulation_Day5_V1Window\TSeries-06142025-1543-001'
# twoP_filename = "TSeries-06142025-1543-001"
# VRlog_filename = "VRlog_JSY038_06142025_05-01-47.txt"

# # # JSY042, miniwalls without landmarks, L2/3, day 3
# # filepath = r'F:\2P\spmod\250612_JSY_JSY042_SpatialModulation_Day3_V1Window\TSeries-06122025-1929-001'
# # twoP_filename = "TSeries-06122025-1929-001"
# # VRlog_filename = "VRlog_JSY038_06122025_07-47-34.txt"

# # # JSY042, miniwalls without landmarks, L2/3, day 1
# # filepath = r'F:\2P\spmod\250610_JSY_JSY042_SpatialModulation_Day1_V1Window\TSeries-06102025-1552-001'
# # twoP_filename = "TSeries-06102025-1552-001"
# # VRlog_filename = "VRlog_JSY038_06102025_04-27-20.txt"

# # # JSY042, miniwalls without landmarks, L2/3
# # filepath = r'F:\2P\spmod\250602_JSY_JSY042_SpatialModulation_Day1_V1Window\TSeries-06022025-1553-001'
# # twoP_filename = "TSeries-06022025-1553-001"
# # VRlog_filename = "VRlog_JSY038_06022025_04-31-58.txt"

# # # JSY042, miniwalls without landmarks (L2/3)
# # filepath = r'D:\V1_SpatialModulation\2p\250523_JSY_JSY042_miniwalls_wolandmarks\TSeries-05232025-1529-001'
# # twoP_filename = "TSeries-05232025-1529-001"
# # VRlog_filename = "VRlog_JSY038_05232025_04-34-23.txt"


# # # JSY038, miniwalls without landmarks, L2/3
# # filepath = r'F:\2P\spmod\250602_JSY_JSY038_SpatialModulation_Day1_V1Window\TSeries-06022025-1553-001'
# # twoP_filename = "TSeries-06022025-1553-001"
# # VRlog_filename = "VRlog_JSY038_06022025_05-57-30.txt"

# VRfilepath = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog"

# VRlog_path = os.path.join(VRfilepath, VRlog_filename)


In [ ]:
# File path to VRlog.txt and 2p data (JSY041 and JSY040 - V1 prism animals)



# # JSY040, miniwalls without landmarks, prism, day 3
# filepath = r'F:\2P\spmod\250622_JSY_JSY040_SpatialModulation_Day3_V1Prism\TSeries-06222025-1550-001'
# twoP_filename = "TSeries-06222025-1550-001"
# VRlog_filename = "VRlog_JSY038_06182025_04-47-38.txt"

# # JSY040, miniwalls without landmarks, prism, day 1
# filepath = r'F:\2P\spmod\250620_JSY_JSY040_SpatialModulation_Day1_V1Prism\TSeries-06202025-1515-001'
# twoP_filename = "TSeries-06202025-1515-001"
# VRlog_filename = "VRlog_JSY038_06222025_04-37-33.txt"




# JSY041, miniwalls without landmarks, prism, day 7
filepath = r'F:\2P\spmod\250622_JSY_JSY041_SpatialModulation_Day7_V1Prism\TSeries-06222025-1550-001'
twoP_filename = "TSeries-06222025-1550-001"
VRlog_filename = "VRlog_JSY038_06222025_04-01-09.txt"

# # JSY041, miniwalls without landmarks, prism, day 5
# filepath = r'F:\2P\spmod\250620_JSY_JSY041_SpatialModulation_Day5_V1Prism\TSeries-06202025-1515-001'
# twoP_filename = "TSeries-06202025-1515-001"
# VRlog_filename = "VRlog_JSY038_06202025_03-34-04.txt"

# # JSY041, miniwalls without landmarks, prism, day 3
# filepath = r'F:\2P\spmod\250618_JSY_JSY041_SpatialModulation_Day3_V1Prism\TSeries-06182025-1641-001'
# twoP_filename = "TSeries-06182025-1641-001"
# VRlog_filename = "VRlog_JSY038_06182025_04-47-38.txt"

# # JSY041, miniwalls without landmarks, prism, day 1
# filepath = r'F:\2P\spmod\250616_JSY_JSY041_SpatialModulation_Day1_V1Prism\TSeries-06162025-1521-001'
# twoP_filename = "TSeries-06162025-1521-001"
# VRlog_filename = "VRlog_JSY038_06162025_04-08-07.txt"

# # JSY041, miniwalls without landmarks
# filepath = r'D:\V1_SpatialModulation\2p\250517_JSY_JSY041_miniwalls_wolandmarks\TSeries-05172025-1438-001'
# twoP_filename = "TSeries-05172025-1438-001"
# VRlog_filename = "VRlog_JSY038_05172025_04-33-39.txt"

VRfilepath = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog"

VRlog_path = os.path.join(VRfilepath, VRlog_filename)


In [ ]:
## previous recordings optimizing the VR track
# # JSY041 BBBB with miniwalls
# filepath = r'D:\V1_SpatialModulation\2p\250511_JSY_JSY041_threecueWminiwalls\TSeries-05112025-1413-001'
# twoP_filename = "TSeries-05112025-1413-001"
# VRlog_filename = "VRlog_JSY038_05112025_04-41-37.txt"

# # JSY038, BBBB
# filepath = r'D:\V1_SpatialModulation\2p\250428_JSY_JSY038_SpatialModulation_BBBB\TSeries-04282025-1500-001'
# twoP_filename = "TSeries-04282025-1500-001"
# VRlog_filename = "VRlog_JSY038_04282025_03-27-04.txt"

# # JSY040, BBBB
# filepath = r'D:\V1_SpatialModulation\2p\250427_JSY_JSY040_SpatialModulation_BBBB\TSeries-04272025-1508-001'
# twoP_filename = "TSeries-04272025-1508-001"
# VRlog_filename = "VRlog_JSY038_04272025_03-38-36.txt"

# # JSY040, BBBB
# filepath = r'D:\V1_SpatialModulation\2p\250501_JSY_JSY040_SpatialModulation_BBBB\TSeries-05012025-1316-002'
# twoP_filename = "TSeries-05012025-1316-002"
# VRlog_filename = "VRlog_JSY038_05012025_02-18-59.txt"

# # JSY041, light intensity graudally changing, coupled movement ++ 3x landmarks, 1st recording
# filepath = r'D:\V1_SpatialModulation\2p\250415_JSY_JSY041_spatialmodulation_3xlandmarks\TSeries-04152025-1559-002'
# twoP_filename = "TSeries-04152025-1559-002"
# VRlog_filename = "VRlog_JSY038_04152025_04-27-07"

# # JSY041, day 1
# filepath = r'D:\V1_SpatialModulation\2p\250320_JSY_JSY041_spatialmodulation\TSeries-03202025-1804-002'
# twoP_filename = "TSeries-03202025-1804-002"
# VRlog_filename = "VRlog_JSY038_03202025_06-56-38.txt"
# 
# # JSY041, day 3
# filepath = r'D:\V1_SpatialModulation\2p\250322_JSY_JSY041_SpatialModulation\TSeries-03222025-1742-002'
# twoP_filename = "TSeries-03222025-1742-002"
# VRlog_filename = "VRlog_JSY038_03222025_06-00-15.txt"

# # # JSY041, day 5
# filepath = r'D:\V1_SpatialModulation\2p\250324_JSY_JSY041_spatialmodulation\TSeries-03242025-1618-002'
# twoP_filename = "TSeries-03242025-1618-002"
# VRlog_filename = "VRlog_JSY038_03242025_05-05-17.txt"


In [ ]:
# Extract animal ID and date from the VR_log_filename
match = re.match(r"VRlog_(JSY\d+)_(\d{8})_\d{2}-\d{2}-\d{2}\.txt", VRlog_filename)
if match:
    animal_id = match.group(1)
    date = match.group(2)
else:
    print("Filename format does not match the expected pattern.")

# Initialize dictionaries to store raw data
twoP_data = {}
VR_data = {}

# Load twoP data
raw_twop_data = TwoP(filepath, twoP_filename)

raw_twop_data.find_files()
twop_dict = raw_twop_data.calc_dFF()

twoP_data['sps'] = twop_dict['spikes_per_sec'].copy()
twoP_data['s2p_spks'] = twop_dict['s2p_spks'].copy()
twoP_data['dFF'] = twop_dict['norm_dFF'].copy()
twoP_data['stat'] = twop_dict['stat'].copy()
twoP_data['ops'] = twop_dict['ops'].copy()

numFrames = np.size(twoP_data['sps'], 1)
numCells = len(twoP_data['stat'])

numFrames = np.size(twoP_data['sps'], 1)

xml_path = os.path.join(filepath, f"{twoP_filename}.xml")
xml_dict = read_xml(xml_path)
t0 = xml_dict["t0"]
abs_time = xml_dict["abs_time"]
rel_time = xml_dict["rel_time"]
framerate = 1/rel_time[1]
print(framerate)

twopT = np.zeros(np.size(abs_time, 0) - 1, dtype=datetime.datetime)
for rep, t in enumerate(abs_time[:-1]):
    twopT[rep] = t0 + datetime.timedelta(seconds=t)

twopT_float = time2float(twopT)
twoP_data['AbsoluteT'] = twopT

im = np.zeros((twoP_data['ops']['Ly'], twoP_data['ops']['Lx']))  # Create an empty image
for n in range(0, numCells):
    ypix = twoP_data['stat'][n]['ypix'][~twoP_data['stat'][n]['overlap']]
    xpix = twoP_data['stat'][n]['xpix'][~twoP_data['stat'][n]['overlap']]
    im[ypix, xpix] = xpix  # Assign xpix values to im for progressive color change along x-axis

# for animal facing 2p computer, image should be rotated so it goes from layer 2/3 to layer 6 (top-bottom)
# for animal facing VR computer, raw image does go from layer 2/3 to layer 6 (top-bottom)
im_rotated = np.rot90(im, k=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(im)
ax1.set_title("raw_image")

ax2.imshow(im_rotated)
ax2.set_title("rotated_image")
plt.show()

# Load VRlog
rawVR_data = []
with open(VRlog_path, "r") as file:
    lines = file.readlines()
    for line in lines[3:]:
        rawVR_data.append(line.strip().split("\t"))

# # Initialize arrays to store extracted data
# absoluteT = []
# elapsedT = []
# events = []
# locations = []
        
# # Extract VR data with special handling for 'p' events
# for line in rawVR_data:
#     if len(line) >= 4:  # Ensure line has at least 4 elements
#         absoluteT.append(line[0])
#         elapsedT.append(float(line[1]))
#         events.append(line[2])
        
#         # Extract location based on event type
#         if line[2] == 'p' and len(line) >= 6:  # Position event with enough elements
#             # For 'p' events, get z-coordinate from 6th column
#             locations.append(float(line[5]))
#         else:
#             # For other events, get location from 4th column
#             locations.append(float(line[3]))
            
# # Convert to numpy arrays
# VR_data = {
#     'absoluteT': np.array(absoluteT),
#     'elapsedT': np.array(elapsedT),
#     'event': np.array(events),
#     'location': np.array(locations)
# }

# # Apply minimum threshold to location data
# VR_data['location'][VR_data['location'] < -40] = -40

# Extract VR data
VR_data['absoluteT'] = np.array([line[0] for line in rawVR_data])
VR_data['elapsedT'] = np.array([float(line[1]) for line in rawVR_data])
VR_data['event'] = np.array([line[2] for line in rawVR_data])
VR_data['location'] = np.array([float(line[3]) for line in rawVR_data])

# for any VR_data['location'] that is less than -40, set it to -40
# VR_data['location'][VR_data['location'] < -40] = -40

# Find the index of the first 's' in VR_data['event']
start_index = np.where(VR_data['event'] == 's')[0][0]

# Erase all elements before the start_index in all VR_data
for key in VR_data.keys():
    VR_data[key] = VR_data[key][start_index:]

# # for every element of VR_data, print first value
# for key in VR_data:
#     print(VR_data[key][0])

print("first time value of VR is", VR_data['absoluteT'][0])
print("first time value of 2p is", twoP_data['AbsoluteT'][0] )

**2. Align the 2p data and treadmill behavior data**

In [ ]:
# Define absolute_t0 as the first element of VR_data['absoluteT']
VR_absolute_t = np.array([datetime.datetime.strptime(t, '%H.%M.%S.%f') for t in VR_data['absoluteT'][0:]])

# Calculate relative_t (time elapsed from absolute_t0)
VR_relative_t = np.array([(t - VR_absolute_t[0]).total_seconds() for t in VR_absolute_t])

# Add twoP_data['AbsoluteT'][0] to each timedelta object to get vrT
VR_relative_t_timedelta = np.array([datetime.timedelta(seconds=t) for t in VR_relative_t])
Aligned_Abs_vrT = twoP_data['AbsoluteT'][0] + VR_relative_t_timedelta

# Calculate relative time points for twoP_data
twop_relativeT = twoP_data['AbsoluteT'] - twoP_data['AbsoluteT'][0]
twop_relativeT = np.array([t.total_seconds() for t in twop_relativeT])
twoP_data['RelativeT'] = twop_relativeT

# Determine which dataset ends first
twop_end_time = twoP_data['AbsoluteT'][-1]
vr_end_time = Aligned_Abs_vrT[-1]

print(f"2P data ends at: {twop_end_time}")
print(f"VR data ends at: {vr_end_time}")

# Create new_VR_data dictionary
new_VR_data = {}

if twop_end_time <= vr_end_time:
    # 2P data ends first - trim VR data to match 2P end time
    print("2P data ends first. Trimming VR data to match.")
    
    # Find the closest value in Aligned_Abs_vrT that is greater than twoP_data['AbsoluteT'][-1]
    valid_indices = Aligned_Abs_vrT > twop_end_time
    if np.any(valid_indices):
        closest_value = Aligned_Abs_vrT[valid_indices][0]
        closest_index = np.where(Aligned_Abs_vrT == closest_value)[0][0]
    else:
        # If no VR timestamp is greater than 2P end time, use all VR data
        closest_index = len(Aligned_Abs_vrT)
    
    # Trim VR data
    new_VR_data['AbsoluteT'] = np.array(Aligned_Abs_vrT)[:closest_index]
    new_VR_data['RelativeT'] = VR_relative_t[:closest_index]
    new_VR_data['event'] = VR_data['event'][:closest_index]
    new_VR_data['location'] = VR_data['location'][:closest_index]
    
    # Keep original 2P data
    final_twoP_data = twoP_data.copy()
    
else:
    # VR data ends first - trim 2P data to match VR end time
    print("VR data ends first. Trimming 2P data to match.")
    
    # Use all VR data
    new_VR_data['AbsoluteT'] = np.array(Aligned_Abs_vrT)
    new_VR_data['RelativeT'] = VR_relative_t
    new_VR_data['event'] = VR_data['event']
    new_VR_data['location'] = VR_data['location']
    
    # Find the closest 2P timestamp that is less than or equal to VR end time
    valid_2p_indices = twoP_data['AbsoluteT'] <= vr_end_time
    if np.any(valid_2p_indices):
        last_valid_2p_index = np.where(valid_2p_indices)[0][-1] + 1  # +1 to include the index
    else:
        last_valid_2p_index = 1  # Keep at least the first point
    
    # Trim 2P data
    final_twoP_data = {}
    for key in twoP_data.keys():
        if key == 'RelativeT':
            # Recalculate relative time for trimmed data
            trimmed_abs_time = twoP_data['AbsoluteT'][:last_valid_2p_index]
            final_twoP_data['RelativeT'] = np.array([(t - trimmed_abs_time[0]).total_seconds() 
                                                   for t in trimmed_abs_time])
        elif isinstance(twoP_data[key], np.ndarray):
            if twoP_data[key].ndim == 1:
                final_twoP_data[key] = twoP_data[key][:last_valid_2p_index]
            elif twoP_data[key].ndim == 2:
                # Check which dimension matches the time dimension
                if twoP_data[key].shape[0] == len(twoP_data['AbsoluteT']):
                    # Time is first dimension: (time, cells)
                    final_twoP_data[key] = twoP_data[key][:last_valid_2p_index, :]
                elif twoP_data[key].shape[1] == len(twoP_data['AbsoluteT']):
                    # Time is second dimension: (cells, time)
                    final_twoP_data[key] = twoP_data[key][:, :last_valid_2p_index]
                else:
                    # If neither dimension matches, keep original
                    print(f"Warning: Could not determine time axis for {key} with shape {twoP_data[key].shape}")
                    final_twoP_data[key] = twoP_data[key]
            else:
                final_twoP_data[key] = twoP_data[key]
        else:
            final_twoP_data[key] = twoP_data[key]

# Apply location clipping
new_VR_data['location'][new_VR_data['location'] < -460] = -460

print(f"Final VR relative time range: {new_VR_data['RelativeT'][0]:.2f} to {new_VR_data['RelativeT'][-1]:.2f} seconds")
print(f"Final 2P relative time range: {final_twoP_data['RelativeT'][0]:.2f} to {final_twoP_data['RelativeT'][-1]:.2f} seconds")

# Interpolate the location at final_twoP_data['RelativeT'] from new_VR_data
interpolated_location = np.interp(final_twoP_data['RelativeT'], 
                                  new_VR_data['RelativeT'], 
                                  new_VR_data['location'])
new_VR_data['interp_location'] = interpolated_location

print(f"Size of interpolated_location: {interpolated_location.shape}")
print(f"Size of new_VR_data['location']: {new_VR_data['location'].shape}")
print(f"Size of final 2P RelativeT: {final_twoP_data['RelativeT'].shape}")

# Plot the interpolated location
plt.figure(figsize=(20, 5))
plt.plot(final_twoP_data['RelativeT'], new_VR_data['interp_location']+100, 
         label="Interpolated Location", alpha=0.7)
plt.plot(new_VR_data['RelativeT'], new_VR_data['location'], 
         label="Original VR Location", alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Location (AU)")
plt.title("Aligned VR Location Data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Optional: Plot 2P data examples if they exist
if 'dFF' in final_twoP_data:
    plt.figure(figsize=(20, 5))
    
    # Check data dimensions and plot accordingly
    if final_twoP_data['dFF'].shape[0] == len(final_twoP_data['RelativeT']):
        # Time is first dimension: (time, cells)
        if final_twoP_data['dFF'].shape[1] > 10:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['dFF'][:, 10])
        else:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['dFF'][:, 0])
    elif final_twoP_data['dFF'].shape[1] == len(final_twoP_data['RelativeT']):
        # Time is second dimension: (cells, time)
        if final_twoP_data['dFF'].shape[0] > 10:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['dFF'][10, :])
        else:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['dFF'][0, :])
    
    plt.xlabel("Time (s)")
    plt.ylabel("dF/F")
    plt.title("Example 2P dF/F Signal")
    plt.grid(True, alpha=0.3)
    print(f"dFF shape: {final_twoP_data['dFF'].shape}, RelativeT shape: {final_twoP_data['RelativeT'].shape}")
    plt.show()

if 'sps' in final_twoP_data:
    plt.figure(figsize=(20, 5))
    
    # Check data dimensions and plot accordingly
    if final_twoP_data['sps'].shape[0] == len(final_twoP_data['RelativeT']):
        # Time is first dimension: (time, cells)
        if final_twoP_data['sps'].shape[1] > 10:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['sps'][:, 10])
        else:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['sps'][:, 0])
    elif final_twoP_data['sps'].shape[1] == len(final_twoP_data['RelativeT']):
        # Time is second dimension: (cells, time)
        if final_twoP_data['sps'].shape[0] > 10:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['sps'][10, :])
        else:
            plt.plot(final_twoP_data['RelativeT'], final_twoP_data['sps'][0, :])
    
    plt.xlabel("Time (s)")
    plt.ylabel("Spikes/s")
    plt.title("Example 2P Spike Rate")
    plt.grid(True, alpha=0.3)
    print(f"sps shape: {final_twoP_data['sps'].shape}, RelativeT shape: {final_twoP_data['RelativeT'].shape}")
    plt.show()

**3a. Find temporal offset, which yields the best alignment between 2p and behavior data**

In [ ]:
# # Define offsets to test (in frames)
# offset_frames_list = [1, 2, 3, 4,5, 7]
# framerate = 1/rel_time[1]  # Use the same framerate as before

# # Dictionary to store results
# offset_results = {}

# # Loop through offsets
# for offset_frames in offset_frames_list:
#     print(f"\n\nTesting offset: {offset_frames} frames ({offset_frames/framerate:.2f} seconds)")
    
#     # Apply offset to original data
#     offset_spike_data = SpikeSmoothing.apply_temporal_offset(twoP_data['sps'], offset_frames)
    
#     # Apply smoothing
#     smoothed = SpikeSmoothing.smooth_spikes(offset_spike_data, framerate, window_ms=500)
    
#     # Process data with trial filtering
#     filtered_spks_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
#         smoothed, 
#         new_VR_data['interp_location'],
#         min_trial_duration_seconds=5, 
#         max_trial_duration_seconds=60,
#         framerate=framerate
#     )
    
#     if n_valid_laps == 0:
#         print(f"No valid laps for offset {offset_frames}")
#         continue
    
#     # single_revolution_VR = 282.415
#     # single_revolution_treadmill = 27.8
#     # single_lap_VR = 1726.99731 ### = 1146 when VR length was 125 at gain = 1.15 
    
#     single_revolution_VR = 282.415
#     single_revolution_treadmill = 27.8
#     single_lap_VR = 1320.645683 ### = 1146 when VR length was 125 at gain = 1.15 
#     single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

#     # single_revolution_VR = 282.415
#     # single_revolution_treadmill = 27.8
#     # single_lap_VR = 1126.0667 ### = 1146 when VR length was 125 at gain = 1.15 
    
#     single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR


#     # Perform spatial assignment
#     spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
#         n_valid_laps,
#         filtered_spks_laps, 
#         filtered_location_laps, 
#         single_lap_treadmill
#     )
    
#     # Apply spatial smoothing
#     smoothed_spatial_activity = SpikeSmoothing.spatial_smooth(spatial_activity, window_cm=10)


#     # Test for reliability
#     reliable_cells, avg_cc, cohens_d, iter_cc, _ = RT.test_cell_reliability(
#         smoothed_spatial_activity,
#         n_shuffles=100,           
#         cc_percentile=90,          
#         cohen_threshold=0.8,       
#         min_cc_threshold=0.25,      
#         min_activity_threshold=0.05, 
#     )

#     # Store results
#     offset_results[offset_frames] = {
#         'reliable_cells': reliable_cells,
#         'reliable_count': np.sum(reliable_cells),
#         'avg_cc': avg_cc,
#         'cohens_d': cohens_d,
#         'spatial_activity': smoothed_spatial_activity,
#         'n_valid_laps': n_valid_laps
#     }

#     # Print summary
#     print(f"Offset {offset_frames}: Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
#     print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
#     print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")
    
#  # Extract metrics for visualization
# valid_offsets = list(offset_results.keys())
# reliable_counts = [offset_results[offset]['reliable_count'] for offset in valid_offsets]
# avg_cc_means = [np.mean(offset_results[offset]['avg_cc'][offset_results[offset]['reliable_cells']]) 
#                if np.sum(offset_results[offset]['reliable_cells']) > 0 else 0
#                for offset in valid_offsets]
# cohens_d_means = [np.mean(offset_results[offset]['cohens_d'][offset_results[offset]['reliable_cells']])
#                  if np.sum(offset_results[offset]['reliable_cells']) > 0 else 0
#                  for offset in valid_offsets]

# # Create figure
# fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

# # Plot reliable cell count
# axes[0].plot(valid_offsets, reliable_counts, 'o-', color='blue', linewidth=2)
# axes[0].set_ylabel('Number of Reliable Cells')
# axes[0].set_title('Effect of Temporal Offset on Cell Reliability Metrics')
# axes[0].grid(True, alpha=0.3)

# # Plot mean correlation coefficient
# axes[1].plot(valid_offsets, avg_cc_means, 'o-', color='green', linewidth=2)
# axes[1].set_ylabel('Mean Correlation Coefficient')
# axes[1].grid(True, alpha=0.3)

# # Plot mean Cohen's D
# axes[2].plot(valid_offsets, cohens_d_means, 'o-', color='red', linewidth=2)
# axes[2].set_xlabel('Offset (frames)')
# axes[2].set_ylabel("Mean Cohen's D")
# axes[2].grid(True, alpha=0.3)

# # Find optimal offset (if results exist)
# if len(valid_offsets) > 0:
#     # Normalize metrics for weighted average
#     norm_reliable = np.array(reliable_counts) / np.max(reliable_counts) if np.max(reliable_counts) > 0 else np.zeros_like(reliable_counts)
#     norm_cc = np.array(avg_cc_means) / np.max(avg_cc_means) if np.max(avg_cc_means) > 0 else np.zeros_like(avg_cc_means)
#     norm_d = np.array(cohens_d_means) / np.max(cohens_d_means) if np.max(cohens_d_means) > 0 else np.zeros_like(cohens_d_means)
    
#     # Weighted sum of normalized metrics
#     combined_metric = (0.2 * norm_reliable + 0.4 * norm_cc + 0.4 * norm_d)
#     # combined_metric = (norm_reliable + norm_cc + norm_d) / 3
#     best_idx = np.argmax(combined_metric)
#     best_offset = valid_offsets[best_idx]
    
#     # Add vertical line at optimal offset
#     for ax in axes:
#         ax.axvline(x=best_offset, color='black', linestyle='--', alpha=0.7)
#         ax.text(best_offset, ax.get_ylim()[1]*0.95, f'Optimal: {best_offset}', 
#                horizontalalignment='center', verticalalignment='top')
    
#     print(f"\nBest offset: {best_offset} frames ({best_offset/framerate:.2f} seconds)")
#     print(f"- Reliable cells: {offset_results[best_offset]['reliable_count']}")
#     print(f"- Mean correlation: {np.mean(offset_results[best_offset]['avg_cc'][offset_results[best_offset]['reliable_cells']]):.3f}")
#     print(f"- Mean Cohen's D: {np.mean(offset_results[best_offset]['cohens_d'][offset_results[best_offset]['reliable_cells']]):.3f}")

# plt.tight_layout()
# plt.show()   

**3b. Smooth the deconvolved traces using a 500 ms Gaussian window**

In [ ]:
best_offset = 0
offset_spike_data = SpikeSmoothing.apply_temporal_offset(twoP_data['sps'], best_offset)
smoothed = SpikeSmoothing.smooth_spikes(offset_spike_data, framerate, window_ms=500)
# smoothed = SpikeSmoothing.smooth_spikes(twoP_data['sps'], framerate, window_ms=500)
twoP_data['smoothed_spks'] = smoothed
# SpikeSmoothing.plot_comparison(twoP_data['s2p_spks'], smoothed, framerate)

**3c. Remove inactive data points**

In [ ]:
### 1) any time points where the treadmill behavior data is not available (i.e. when the animal is not moving)
### 2) any trial that is taking less than 5 seconds or more than 30 seconds to complete
min_trial_duration_seconds = 10
max_trial_duration_seconds = 50

filtered_spks_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
    twoP_data['smoothed_spks'], 
    new_VR_data['interp_location'],
    min_trial_duration_seconds = min_trial_duration_seconds, 
    max_trial_duration_seconds = max_trial_duration_seconds,
    framerate=framerate
)

**4. Discretize spatial data (1cm/bin) and assign 2p data to a corresponding bin**

In [ ]:
# # when VR length was 300 at gain = 1.15 - 25/03/20 
single_revolution_VR = 282.415
single_revolution_treadmill = 27.8
# single_lap_VR = 1726.99731 ### = 1146 when VR length was 125 at gain = 1.15 
# single_lap_VR = 1320.645683 ### = 1146 when VR length was 125 at gain = 1.15 
single_lap_VR = 1320.645683 ### = 1146 when VR length was 125 at gain = 1.15 
single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# # when VR length was 125 at gain = 1.15 
# single_revolution_VR = 285.9317
# single_revolution_treadmill = 27.8
# single_lap_VR = 1146 ### = 1146 when VR length was 125 at gain = 1.15 
# single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# Then perform spatial assignment on the filtered data
spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
    n_valid_laps,
    filtered_spks_laps, 
    filtered_location_laps, 
    single_lap_treadmill
)

window_cm = 5
smoothed_spatial_activity = SpikeSmoothing.spatial_smooth(spatial_activity, window_cm=window_cm)
normalized_spatial_activity = RT.normalize_spatial_activity(smoothed_spatial_activity)

# remove beginning and end of spatial data (beginning is when light is brigtening and end is when light is dimming + tunnel appears)
spatial_activity = spatial_activity[:, :, 9:121]
spatial_bins = spatial_bins[9:122]
trial_averaged_activity = trial_averaged_activity[:, 9:121]
bin_centers = bin_centers[9:121]
normalized_spatial_activity = normalized_spatial_activity[:, :, 9:121]

**5. Test for reliability for individual cells (uses Pearson CC and cohen’s D)**

In [ ]:
# # Run the analysis
# reliable_cells, avg_cc, cohens_d, iter_cc, _ = RT.test_cell_reliability(
#     smoothed_spatial_activity,
#     n_shuffles=100,           # Use 1000+ for final analysis
#     cc_percentile=95,          # 90th percentile threshold for CC
#     cohen_threshold=1.2,       # Medium-large effect size
#     min_cc_threshold=0.3,      # Minimum correlation required
#     min_activity_threshold=0.1 # Minimum activity level (relative to max)
# )

# # Print summary
# print(f"Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
# print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
# print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")

def evaluate_pattern_similarity(spatial_activity, min_pattern_corr=0.4, peak_distance_threshold=5):
    """
    Evaluates both correlation and pattern similarity between odd and even trials.
    
    Parameters:fro
    -----------
    spatial_activity : numpy.ndarray
        Activity matrix (n_cells x n_trials x n_spatial_bins)
    min_pattern_corr : float
        Minimum correlation threshold for pattern similarity
    peak_distance_threshold : int
        Maximum allowed distance between peaks in odd and even trials (in bins)
        
    Returns:
    --------
    pattern_reliable : numpy.ndarray
        Boolean array of cells with consistent patterns
    odd_even_corr : numpy.ndarray
        Correlation between odd and even trials for each cell
    peak_distances : numpy.ndarray
        Distance between peaks in odd and even trials for each cell
    """
    n_cells, n_trials, n_bins = spatial_activity.shape
    
    # Initialize output arrays
    pattern_reliable = np.zeros(n_cells, dtype=bool)
    odd_even_corr = np.zeros(n_cells)
    peak_distances = np.zeros(n_cells)
    
    # Separate odd and even trials
    odd_trials = np.arange(0, n_trials, 2)
    even_trials = np.arange(1, n_trials, 2)
    
    for cell in range(n_cells):
        # Calculate mean activity for odd and even trials
        odd_mean = np.mean(spatial_activity[cell, odd_trials], axis=0)
        even_mean = np.mean(spatial_activity[cell, even_trials], axis=0)
        
        # Calculate correlation between odd and even mean activity patterns
        odd_even_corr[cell] = np.corrcoef(odd_mean, even_mean)[0, 1]
        
        # Find peaks in odd and even trials
        odd_peak = np.argmax(odd_mean)
        even_peak = np.argmax(even_mean)
        
        # Calculate shortest distance between peaks (accounting for circular track)
        raw_distance = abs(odd_peak - even_peak)
        circular_distance = min(raw_distance, n_bins - raw_distance)
        peak_distances[cell] = circular_distance
        
        # Check if patterns are similar (high correlation and consistent peak location)
        if (odd_even_corr[cell] >= min_pattern_corr and 
            peak_distances[cell] <= peak_distance_threshold):
            pattern_reliable[cell] = True
    
    return pattern_reliable, odd_even_corr, peak_distances

def combined_reliability_test(spatial_activity, n_shuffles=1000, 
                             cc_percentile=95, cohen_threshold=0.5,
                             min_cc_threshold=0.2, min_activity_threshold=0.1,
                             min_pattern_corr=0.4, peak_distance_threshold=5):
    """
    Combined reliability test checking both trial-to-trial reliability and odd-even pattern similarity
    """
    # Run original reliability test
    reliable_cells, avg_cc, cohens_d, iter_cc, norm_activity = RT.test_cell_reliability(
        spatial_activity, n_shuffles, cc_percentile, cohen_threshold,
        min_cc_threshold, min_activity_threshold
    )
    
    # Run pattern similarity test
    pattern_reliable, odd_even_corr, peak_distances = evaluate_pattern_similarity(
        spatial_activity, min_pattern_corr, peak_distance_threshold
    )
    
    # Combine results (cells must pass both tests)
    combined_reliable = reliable_cells & pattern_reliable
    
    return (combined_reliable, reliable_cells, pattern_reliable, 
            avg_cc, cohens_d, odd_even_corr, peak_distances, norm_activity)
    
# Run the analysis
combined_reliable, reliable_cells, _, avg_cc, cohens_d, _, _, _ = combined_reliability_test(
    smoothed_spatial_activity,
    n_shuffles=100,           # Use 1000+ for final analysis
    cc_percentile=90,          # 90th percentile threshold for CC
    cohen_threshold=1,       # Medium-large effect size
    min_cc_threshold=0.3,      # Minimum correlation required
    min_activity_threshold=0.1, # Minimum activity level (relative to max)
    min_pattern_corr=0.3, 
    peak_distance_threshold=5
)

print(f"Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
print(f"Found {np.sum(combined_reliable)} combined_reliable -- reliable in both even and odd trials -- cells out of {len(combined_reliable)}")


In [ ]:
# # Option 1.1 Plot the average activity oin a grid layout
# reliable_avg = {}

# for i in np.where(reliable_cells)[0]:
#     reliable_avg[i] = np.mean(spatial_activity[i, :, :], axis=0)

# # plot average activity of first 25 reliable cells in a 5 x 5 subplots
# fig, axs = plt.subplots(5, 5, figsize=(20, 20))
# reliable_indices = list(reliable_avg.keys())[:25]
# for idx, i in enumerate(reliable_indices):
#     ax = axs[idx // 5, idx % 5]
#     ax.plot(bin_centers, reliable_avg[i])
#     ax.set_title(f'Cell {i}')
#     ax.set_xlabel('Position')
#     ax.set_ylabel('Activity')

# # set a title over all plots
# fig.suptitle('Average spikes of example reliable cells', fontsize=16)
# plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust the rect parameter to add space at the top
# plt.show()

# # Option 1.2: Plot heatmaps in a grid layout (eg. 20 cells in a 5×4 grid)
# fig1 = RT.plot_reliable_cells_grid(
#     normalized_spatial_activity,
#     reliable_cells,
#     max_cells=20,                # Show up to 20 reliable cells
#     avg_cc=avg_cc,               # Optional correlation coefficients
#     cohen_d=cohens_d,            # Optional Cohen's D values
#     normalize=False,              # Apply normalization
#     n_rows=5,                    # 5 rows
#     n_cols=4                     # 4 columns
# )
# plt.suptitle('Spikes of reliable cells', fontsize=15)
# plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top

# plt.show()

# Option 2: Plot with both heatmap and trial-averaged activity
fig2 = RT.plot_reliable_cells_side_by_side(
    normalized_spatial_activity,
    reliable_cells,
    # combined_reliable,
    max_cells=10,              # Show up to 10 reliable cells
    # max_cells=np.sum(reliable_cells),                # Show up to 10 reliable cells
    avg_cc=avg_cc,               # Optional correlation coefficients
    cohen_d=cohens_d,            # Optional Cohen's D values
    normalize=False               # Apply normalization
)
plt.suptitle('Spikes of reliable cells', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top
plt.show()

# # Option 3: Plot waterfall plots
# RT.plot_reliable_cells_waterfall(normalized_spatial_activity, reliable_cells, max_cells=np.sum(reliable_cells))
# plt.show()

In [ ]:
# Option 2: Plot with both heatmap and trial-averaged activity
fig2 = RT.plot_reliable_cells_side_by_side(
    normalized_spatial_activity,
    # reliable_cells,
    combined_reliable,
    max_cells=10,              # Show up to 10 reliable cells
    # max_cells=np.sum(combined_reliable),                # Show up to 10 reliable cells
    avg_cc=avg_cc,               # Optional correlation coefficients
    cohen_d=cohens_d,            # Optional Cohen's D values
    normalize=False               # Apply normalization
)
plt.suptitle('Spikes of reliable cells', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top
plt.show()

**6. Response Plot - plotting activity of all responsive cells (cross validation – split trials in half)**

In [ ]:
miniwall_location = [16, 45, 74, 103]

In [ ]:
fig1, sorted_indices = RV.create_response_plot(normalized_spatial_activity, reliable_cells, clim=(0, 1))  # Optional: manually set contrast limits for stronger effect
# Add vertical lines only to the heatmap (right subplot)
if len(fig1.axes) >= 2:  # Make sure we have at least 2 subplots
    heatmap_ax = fig1.axes[1]  # The heatmap is typically the second axis
    
    # Add red vertical lines at positions 55, 95, 135 and label as miniwall("D" as in divider)
    for i in miniwall_location:
        heatmap_ax.axvline(x=i, color='red', linestyle='--', linewidth=2, alpha=0.8)
        heatmap_ax.text(i, 20, 'D', color='red', 
                       fontsize=20, ha='center', va='center')
        
    #     # Add green vertical lines at positions 55, 95, 135 and label as landmark B
    # for i in [35, 65, 95]:
    #     heatmap_ax.axvline(x=i, color='green',linestyle='--', linewidth=2, alpha=0.9)
    #     heatmap_ax.text(i, 20, 'B', color='green',
    #                    fontsize=20, ha='center', va='center')
plt.show()

fig2, sorted_indices = RV.create_response_plot(normalized_spatial_activity, combined_reliable, clim=(0, 1))  # Optional: manually set contrast limits for stronger effect
if len(fig2.axes) >= 2:  # Make sure we have at least 2 subplots
    heatmap_ax = fig2.axes[1]  # The heatmap is typically the second axis
    
    # Add red vertical lines at positions 55, 95, 135 and label as landmark B
    for i in miniwall_location:
        heatmap_ax.axvline(x=i, color='red', linestyle='--', linewidth=2, alpha=0.8)
        heatmap_ax.text(i, 20, 'D', color='red', 
                       fontsize=20, ha='center', va='center')
        
    #     # Add green vertical lines at positions 55, 95, 135 and label as landmark B
    # for i in [35, 65, 95]:
    #     heatmap_ax.axvline(x=i, color='green',linestyle='--', linewidth=2, alpha=0.9)
    #     heatmap_ax.text(i, 20, 'B', color='green',
    #                    fontsize=20, ha='center', va='center')
plt.show()

**7. Calculate Spatial Modulation Index (SMI)**

In [ ]:
print(trial_averaged_activity.shape)
fig, axs = plt.subplots(8, 5, figsize=(30, 15))  # Define axs using plt.subplots

for i, n in enumerate(range(100, 140)):
    ax = axs[i // 5, i % 5]
    ax.plot(trial_averaged_activity[n, :])  # Fix indexing syntax
    ax.set_title(f"Cell {n}")  # Use ax.set_title for individual subplots
plt.tight_layout()
plt.show()

plt.plot(trial_averaged_activity[136, :])

for i in miniwall_location:
    plt.axvline(x=i, color='red', linestyle='--', alpha=0.5)

In [ ]:
# ## delete spatial bins in the beginning and end of the spatial activity data

# # smoothed_spatial_activity1 = smoothed_spatial_activity[:, :, 20:120]
# # normalized_spatial_activity1 = normalized_spatial_activity[:, :, 20:120]
# # trial_averaged_activity1 = trial_averaged_activity[:, 20:120]
# # bin_centers1 = bin_centers[20:120]
# # spatial_bins1 = spatial_bins[20:120]

# # smoothed_spatial_activity1 = smoothed_spatial_activity[:, :, 20:160]
# # trial_averaged_activity1 = trial_averaged_activity[:, 20:160]
# # bin_centers1 = bin_centers[20:160]
# # spatial_bins1 = spatial_bins[20:160]

# # smoothed_spatial_activity1 = np.concatenate((smoothed_spatial_activity[:, :, 20:70], smoothed_spatial_activity[:, :, 110:160]), axis=2)
# # trial_averaged_activity1 = np.concatenate((trial_averaged_activity[:, 20:70], trial_averaged_activity[:, 110:160]), axis=1)
# # bin_centers1 = np.concatenate((bin_centers[20:70], bin_centers[110:160]))
# # spatial_bins1 = np.concatenate((spatial_bins[20:70], spatial_bins[110:160]))
# # print(bin_centers1.shape)

# # smoothed_spatial_activity1 = smoothed_spatial_activity[:, :, 10:100]
# # normalized_spatial_activity1 = normalized_spatial_activity[:, :, 10:100]
# # alt_normalizeddata1 = alt_normalizeddata[:, :, 10:100]

# # trial_averaged_activity1 = trial_averaged_activity[:, 10:100]
# # bin_centers1 = bin_centers[10:100]
# # spatial_bins1 = spatial_bins[10:100]

# smoothed_spatial_activity1 = smoothed_spatial_activity
# normalized_spatial_activity1 = normalized_spatial_activity

# trial_averaged_activity1 = trial_averaged_activity
# bin_centers1 = bin_centers
# spatial_bins1 = spatial_bins


In [ ]:
# Step 1: Shift to start at 0
shifted_centers = bin_centers - np.min(bin_centers)

# Step 2: Scale to match the actual physical distance (0 to 110cm)
actual_corridor_length = np.size(spatial_bins)  # cm
unity_corridor_length = np.max(shifted_centers)
scaled_bin_centers = shifted_centers * (actual_corridor_length / unity_corridor_length)

# Set parameters
segment_distance = 29 # Distance between visually identical positions
exclude_boundary_cm = 0  # Border to exclude

results = SMI.analyze_spatial_modulation_BBBB(normalized_spatial_activity, scaled_bin_centers, reliable_cells, avg_cc =avg_cc, cohens_d =cohens_d,
                                        segment_distance=segment_distance, exclude_boundary_cm=exclude_boundary_cm)
plt.show()


# Extract the SMI values for valid cells
SMI_values = results['smi_results']['SMI']
reliable_valid_cells = results['smi_results']['reliable_valid_cells']
reliable_valid_SMI = SMI_values[reliable_valid_cells]

# Remove any NaN or Inf values if present
reliable_valid_SMI = reliable_valid_SMI[~np.isnan(reliable_valid_SMI) & ~np.isinf(reliable_valid_SMI)]
print("")
print(f"Number of total cells: " f"{len(SMI_values)}" " and number of reliable and valid cells: " f"{len(reliable_valid_SMI)}")

# Calculate summary statistics
median_SMI = np.median(reliable_valid_SMI)
mad_SMI = stats.median_abs_deviation(reliable_valid_SMI)
print(f"Median SMI ± MAD: {median_SMI:.2f} ± {mad_SMI:.2f}")

# Statistical test (Wilcoxon signed-rank test against 0)
stat, p_value = stats.wilcoxon(reliable_valid_SMI)
print(f"Wilcoxon test against SMI=0: p-value = {p_value:.2e}")

# Create cumulative distribution plot with full SMI range (including negative values)
plt.figure(figsize=(8, 6))
x_sorted = np.sort(reliable_valid_SMI)  # Keep original SMI values (including negatives)
y_cumulative = np.arange(1, len(x_sorted) + 1) / len(x_sorted)
plt.plot(x_sorted, y_cumulative, 'k-', linewidth=2)

# Add reference lines
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7)

plt.xlabel('Spatial modulation index')
plt.ylabel('Cumul. probability')
plt.title('Cumulative Distribution of Spatial Modulation Index')
plt.xlim(-1, 1)
plt.ylim(0, 1)
plt.grid(False)
plt.tight_layout()
plt.show()

# Print proportion of cells with different modulation patterns
prop_positive = np.mean(reliable_valid_SMI > 0)
prop_negative = np.mean(reliable_valid_SMI < 0)
prop_strong_pos = np.mean(reliable_valid_SMI > 0.5)
print(f"Proportion of cells with positive modulation (SMI > 0): {prop_positive:.2f} ({prop_positive*100:.1f}%)")
print(f"Proportion of cells with negative modulation (SMI < 0): {prop_negative:.2f} ({prop_negative*100:.1f}%)")
print(f"Proportion of cells with strong positive modulation (SMI > 0.5): {prop_strong_pos:.2f} ({prop_strong_pos*100:.1f}%)")

In [ ]:
# SMI_values = results['smi_results']['SMI']
# reliable_valid_cells = results['smi_results']['reliable_valid_cells']
# reliable_valid_SMI = SMI_values[reliable_valid_cells]


print(SMI_values.shape)
print(reliable_valid_SMI.shape)
print(reliable_valid_SMI)

In [ ]:
def analyze_and_plot_SMI(results, save_path=None, title_suffix=""):
    """
    Analyze SMI results and create pie chart visualization
    
    Parameters:
    -----------
    results : dict
        Output from calculate_SMI_BBBB function (may be nested under 'smi_results')
    save_path : str, optional
        Directory to save the plot
    title_suffix : str, optional
        Additional text for plot title
    """    
    
    # Handle nested structure
    if 'smi_results' in results:
        smi_data = results['smi_results']
    else:
        smi_data = results
    
    # Extract SMI values for reliable and valid cells
    if smi_data['reliable_valid_cells'] is not None:
        # Ensure the reliable_valid_cells mask matches the SMI array size
        reliable_mask = smi_data['reliable_valid_cells']
        if len(reliable_mask) != len(smi_data['SMI']):
            print(f"Warning: reliable_valid_cells mask size ({len(reliable_mask)}) doesn't match SMI array size ({len(smi_data['SMI'])})")
            # Use only valid_cells instead
            valid_mask = smi_data['valid_cells']
            smi_values = smi_data['SMI'][valid_mask]
            total_cells = np.sum(valid_mask)
            print(f"Using valid cells only due to size mismatch: {total_cells}")
        else:
            valid_mask = reliable_mask
            smi_values = smi_data['SMI'][valid_mask]
            total_cells = np.sum(valid_mask)
            print(f"Using reliable & valid cells: {total_cells}")
    else:
        valid_mask = smi_data['valid_cells']
        smi_values = smi_data['SMI'][valid_mask]
        total_cells = np.sum(valid_mask)
        print(f"Using valid cells only: {total_cells}")
    
    # Define SMI categories
    categories = ['Strongly modulated\n(SMI > 0.5)', 
                  'Moderately modulated\n(0.2 < SMI ≤ 0.5)', 
                  'Weakly modulated\n(0 < SMI ≤ 0.2)', 
                  'Non-modulated\n(|SMI| ≤ 0.05)', 
                  'Inverted modulation\n(SMI < 0)']
    
    # Categorize cells based on SMI values
    strongly_modulated = np.sum(smi_values > 0.5)
    moderately_modulated = np.sum((smi_values > 0.2) & (smi_values <= 0.5))
    weakly_modulated = np.sum((smi_values > 0.05) & (smi_values <= 0.2))  # Changed from 0 to 0.05
    non_modulated = np.sum(np.abs(smi_values) <= 0.05)
    inverted_modulated = np.sum(smi_values < -0.05)  # Added threshold for negative values
    
    cell_counts = [strongly_modulated, moderately_modulated, weakly_modulated, 
                   non_modulated, inverted_modulated]
    
    # Calculate percentages
    percentages = [(count/total_cells)*100 for count in cell_counts]
    
    # Print summary statistics
    print(f"\nSMI Distribution Analysis:")
    print(f"Total reliable & valid cells: {total_cells}")
    print(f"Strongly modulated (SMI > 0.5): {strongly_modulated} ({percentages[0]:.1f}%)")
    print(f"Moderately modulated (0.2 < SMI ≤ 0.5): {moderately_modulated} ({percentages[1]:.1f}%)")
    print(f"Weakly modulated (0.05 < SMI ≤ 0.2): {weakly_modulated} ({percentages[2]:.1f}%)")
    print(f"Non-modulated (|SMI| ≤ 0.05): {non_modulated} ({percentages[3]:.1f}%)")
    print(f"Inverted modulation (SMI < -0.05): {inverted_modulated} ({percentages[4]:.1f}%)")
    print(f"Mean SMI: {np.mean(smi_values):.3f} ± {np.std(smi_values):.3f}")
    print(f"Median SMI: {np.median(smi_values):.3f}")
    print(f"SMI range: [{np.min(smi_values):.3f}, {np.max(smi_values):.3f}]")
    
    # Create save directory if specified
    if save_path:
        os.makedirs(save_path, exist_ok=True)
    
    # Define colors for each category
    colors = ['#ff4444', '#ff8844', '#ffcc44', '#cccccc', '#4444ff']
    
    # Only include categories with non-zero counts for cleaner visualization
    non_zero_indices = [i for i, count in enumerate(cell_counts) if count > 0]
    filtered_counts = [cell_counts[i] for i in non_zero_indices]
    filtered_categories = [categories[i] for i in non_zero_indices]
    filtered_colors = [colors[i] for i in non_zero_indices]
    filtered_percentages = [percentages[i] for i in non_zero_indices]
    
    # Create pie chart with custom labeling strategy
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Calculate which segments are too small for internal labels (less than 5%)
    small_segments = [pct < 5.0 for pct in filtered_percentages]
    
    # Create pie chart without automatic labels
    wedges, texts, autotexts = ax.pie(filtered_counts, 
                                      labels=None,
                                      colors=filtered_colors, 
                                      autopct='',  # We'll add labels manually
                                      startangle=90)
    
    # First pass: collect all outside labels and their positions
    outside_label_info = []
    for i, (wedge, count, pct, is_small) in enumerate(zip(wedges, filtered_counts, filtered_percentages, small_segments)):
        if is_small:
            angle = (wedge.theta2 + wedge.theta1) / 2
            outside_label_info.append({
                'wedge': wedge,
                'count': count,
                'pct': pct,
                'angle': angle,
                'index': i
            })
    
    # Sort outside labels by angle
    outside_label_info.sort(key=lambda x: x['angle'])
    
    # Add labels manually - either inside or with angled leader lines
    for i, (wedge, count, pct, is_small) in enumerate(zip(wedges, filtered_counts, filtered_percentages, small_segments)):
        angle = (wedge.theta2 + wedge.theta1) / 2  # Middle angle of the wedge
        
        label_text = f'{pct:.1f}% ({count} cells)'
        
        if is_small:
            # Find this label's position in the sorted outside labels
            label_position = next((j for j, info in enumerate(outside_label_info) if info['index'] == i), 0)
            
            # Start from the edge of the pie
            x_edge = 1.0 * np.cos(np.radians(angle))
            y_edge = 1.0 * np.sin(np.radians(angle))
            
            # Create different angles for each outside label to avoid overlap
            # Base angle extends outward from the pie segment
            base_extension_angle = angle
            
            # Modify the angle slightly for each label to create height differences
            if len(outside_label_info) > 1:
                # Create angular offsets: spread labels by ±15 degrees from base angle
                max_offset = 15  # degrees
                offset_per_label = (2 * max_offset) / (len(outside_label_info) - 1) if len(outside_label_info) > 1 else 0
                angle_offset = (label_position * offset_per_label) - max_offset
                final_angle = base_extension_angle + angle_offset
            else:
                final_angle = base_extension_angle
            
            # Fixed line length for consistency - extending OUTWARD from pie edge
            line_length = 0.6
            
            # Calculate end position - this should extend OUTWARD from the pie edge
            x_outer = x_edge + line_length * np.cos(np.radians(final_angle))
            y_outer = y_edge + line_length * np.sin(np.radians(final_angle))
            
            # Ensure we're extending outward, not inward
            # Check if the line goes toward center (distance from origin should increase)
            distance_edge = np.sqrt(x_edge**2 + y_edge**2)
            distance_outer = np.sqrt(x_outer**2 + y_outer**2)
            
            if distance_outer <= distance_edge:
                # If line goes inward, force it outward by reversing the direction
                x_outer = x_edge - line_length * np.cos(np.radians(final_angle))
                y_outer = y_edge - line_length * np.sin(np.radians(final_angle))
            
            # If the label would be too high (near title), adjust it downward slightly
            max_y_allowed = 0.8  # Keep labels below this to avoid title
            if y_outer > max_y_allowed:
                # Scale down the y position while keeping x the same
                y_outer = max_y_allowed
                
            # If multiple labels are close together, add small spacing adjustments
            if len(outside_label_info) > 1:
                # Add small vertical offset to prevent overlap between nearby labels
                vertical_offset = (label_position - (len(outside_label_info) - 1) / 2) * 0.1
                y_outer += vertical_offset
            
            # Draw single straight leader line
            ax.plot([x_edge, x_outer], [y_edge, y_outer], 'k-', linewidth=1, alpha=0.7)
            
            # Determine text alignment based on final position
            ha_align = 'left' if x_outer > 0 else 'right'
            
            # Add text with background
            ax.text(x_outer, y_outer, label_text, 
                   fontsize=10, fontweight='bold',
                   ha=ha_align,
                   va='center',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))
        else:
            # For large segments, place text in center
            x_center = 0.6 * np.cos(np.radians(angle))
            y_center = 0.6 * np.sin(np.radians(angle))
            
            ax.text(x_center, y_center, label_text,
                   fontsize=11, fontweight='bold', color='white',
                   ha='center', va='center')
    
    # Create legend
    legend = ax.legend(wedges, filtered_categories,
                      title="SMI Categories",
                      loc="center left",
                      bbox_to_anchor=(1.05, 0.5),
                      fontsize=12,
                      title_fontsize=13,
                      frameon=True,
                      fancybox=True,
                      shadow=True)
    
    # Fixed title formatting with more spacing
    # Main title in bold - positioned higher
    ax.text(0.5, 1.15, f'SMI Modulation Distribution{title_suffix}', 
            transform=ax.transAxes, fontsize=16, fontweight='bold', 
            ha='center', va='bottom')
    
    # Subtitle with total cells - positioned higher
    ax.text(0.5, 1.08, f'Total Cells: {total_cells}', 
            transform=ax.transAxes, fontsize=14, 
            ha='center', va='bottom')
    
    # Adjust the pie chart position to create more space
    ax.set_position([0.1, 0.1, 0.6, 0.75])  # [left, bottom, width, height]
    
    plt.tight_layout()
    
    # if save_path:
    #     save_file = os.path.join(save_path, 'SMI_modulation_pie_chart.png')
    #     plt.savefig(save_file, dpi=300, bbox_inches='tight')
    #     print(f"\\nPie chart saved to: {save_file}")
    
    plt.show()
    
    return {
        'total_cells': total_cells,
        'cell_counts': cell_counts,
        'percentages': percentages,
        'categories': categories,
        'smi_values': smi_values,
        'statistics': {
            'mean': np.mean(smi_values),
            'std': np.std(smi_values),
            'median': np.median(smi_values),
            'min': np.min(smi_values),
            'max': np.max(smi_values)
        }
    }

def plot_SMI_histogram(results, save_path=None, bins=50):
    """
    Create histogram of SMI values
    """
    # Handle nested structure
    if 'smi_results' in results:
        smi_data = results['smi_results']
    else:
        smi_data = results
    
    if smi_data['reliable_valid_cells'] is not None:
        # Ensure the reliable_valid_cells mask matches the SMI array size
        reliable_mask = smi_data['reliable_valid_cells']
        if len(reliable_mask) != len(smi_data['SMI']):
            print(f"Warning: reliable_valid_cells mask size ({len(reliable_mask)}) doesn't match SMI array size ({len(smi_data['SMI'])})")
            # Use only valid_cells instead
            valid_mask = smi_data['valid_cells']
            smi_values = smi_data['SMI'][valid_mask]
        else:
            valid_mask = reliable_mask
            smi_values = smi_data['SMI'][valid_mask]
    else:
        valid_mask = smi_data['valid_cells']
        smi_values = smi_data['SMI'][valid_mask]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create histogram
    n, bins_edges, patches = ax.hist(smi_values, bins=bins, alpha=0.7, color='skyblue', edgecolor='black')
    
    # Add vertical lines for category boundaries
    ax.axvline(x=0.5, color='red', linestyle='--', label='Strong modulation threshold')
    ax.axvline(x=0.2, color='orange', linestyle='--', label='Moderate modulation threshold')
    ax.axvline(x=0.05, color='gray', linestyle='--', label='Weak modulation threshold')
    ax.axvline(x=-0.05, color='gray', linestyle='--')
    ax.axvline(x=0, color='black', linestyle='-', alpha=0.5, label='Zero modulation')
    
    ax.set_xlabel('SMI Value', fontsize=12)
    ax.set_ylabel('Number of Cells', fontsize=12)

    ax.set_title(f'Distribution of SMI Values. Total Cells: {len(smi_values)}\nMean: {np.mean(smi_values):.3f}, Std: {np.std(smi_values):.3f}, Median: {np.median(smi_values):.3f}', fontsize=14, fontweight='bold')

    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # if save_path:
    #     save_file = os.path.join(save_path, 'SMI_histogram.png')
    #     plt.savefig(save_file, dpi=300, bbox_inches='tight')
    #     print(f"Histogram saved to: {save_file}")
    
    plt.show()

# Create visualizations
save_path = r"C:\\Users\\jasmineyeo\\Desktop"

# Pie chart analysis
analysis_results = analyze_and_plot_SMI(results, save_path=save_path, title_suffix=" - miniwalls only")

# Histogram 
plot_SMI_histogram(results, save_path=save_path)


In [ ]:
import math
preferred_positions = results['smi_results']['preferred_positions']
reliable_valid_cells = results['smi_results']['reliable_valid_cells']
# find index of reliable_valid_cells in reliable_cells
reliable_valid_cells_indices = np.where(reliable_valid_cells)[0]

preferred_position_reliable_valid = np.zeros_like(reliable_valid_cells_indices, dtype=float)
for i in range(len(reliable_valid_cells_indices)):
    preferred_position_reliable_valid[i] = preferred_positions[reliable_valid_cells_indices[i]]
    
plt.hist(preferred_position_reliable_valid, bins=20, color='black', alpha=0.8)

max_value = round(np.max(preferred_positions))
boundaries = np.arange(math.floor(max_value/4), max_value + 1, round(max_value/4))  # Adjust the bin size as needed

preferred_first = np.where((preferred_position_reliable_valid > 0) & (preferred_position_reliable_valid <= boundaries[0]))[0]
preferred_second = np.where((preferred_position_reliable_valid > boundaries[0]) & (preferred_position_reliable_valid <=  boundaries[1]))[0]
preferred_third = np.where((preferred_position_reliable_valid >  boundaries[1]) & (preferred_position_reliable_valid <=  boundaries[2]))[0]
preferred_fourth = np.where((preferred_position_reliable_valid >  boundaries[2]) & (preferred_position_reliable_valid <=  boundaries[3]))[0]

plt.axvline(x=boundaries[0], color='red', linestyle='--', alpha=0.5, label=preferred_first)
plt.axvspan(0,boundaries[0],  color='red', alpha=0.2, label='Preferred 1st')
plt.text(boundaries[0]/2, 20, str(np.size(preferred_first)), color='red', fontsize=10, ha='center', va='center', fontweight='bold')

plt.axvline(x=boundaries[1], color='green', linestyle='--', alpha=0.5, label=preferred_second)
plt.axvspan(boundaries[0],boundaries[1],  color='green', alpha=0.2, label='Preferred 2nd')
plt.text((boundaries[0] + boundaries[1])/2, 20, str(np.size(preferred_second)), color='green', fontsize=10, ha='center', va='center', fontweight='bold')

plt.axvline(x=boundaries[2], color='blue', linestyle='--', alpha=0.5, label=preferred_third)
plt.axvspan(boundaries[1],boundaries[2],  color='blue', alpha=0.2, label='Preferred 3rd')
plt.text((boundaries[1] + boundaries[2])/2, 20, str(np.size(preferred_third)), color='blue', fontsize=10, ha='center', va='center', fontweight='bold')

plt.axvline(x=boundaries[3], color='purple', linestyle='--', alpha=0.5, label=preferred_fourth)

plt.axvspan(boundaries[2],boundaries[3],  color='purple', alpha=0.2, label='Preferred 4th')
plt.text((boundaries[2] + boundaries[3])/2, 20, str(np.size(preferred_fourth)), color='purple', fontsize=10, ha='center', va='center', fontweight='bold')
plt.xlabel('Preferred position (cm)', fontsize=12)
plt.ylabel('Number of cells', fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.title('Histogram of preferred positions for reliable cells', fontsize=14)


**8. Layer-specific analysis (reliability and SMI)**

In [ ]:
# Extract the median coordinates of each cell
med_coords = np.array([cell['med'] for cell in twoP_data['stat']])
layer_cells, layer_boundaries = SMI_Layer.identify_layers(med_coords)
SMI_Layer.plot_layer_distribution(med_coords, layer_cells, combined_reliable,im)
plt.show()

In [ ]:
layer_results, layer_cells = SMI_Layer.run_layer_SMI_analysis(
    smi_results=results['smi_results'],
    reliable_cells=reliable_valid_cells,
    med_coords=med_coords,
    layer_cells=layer_cells,
    normalized_spatial_activity=normalized_spatial_activity,
    bin_centers=scaled_bin_centers
)

In [ ]:
print((layer_results['L2/3']['SMI']))

In [ ]:
print((layer_results['L4']['SMI']))

In [ ]:
print((layer_results['L5']['SMI']))

In [ ]:
print((layer_results['L6']['SMI']))

In [ ]:
# visualize top cells in each layer
SMI_Layer.visualize_top_cells(
    normalized_spatial_activity,
    layer_results,
    'L2/3',  # Choose a layer
    scaled_bin_centers,
    top_n=20
)

SMI_Layer.visualize_top_cells(
    normalized_spatial_activity,
    layer_results,
    'L4',  # Choose a layer
    scaled_bin_centers,
    top_n=20
)

SMI_Layer.visualize_top_cells(
    normalized_spatial_activity,
    layer_results,
    'L5',  # Choose a layer
    scaled_bin_centers,
    top_n=20
)

SMI_Layer.visualize_top_cells(
    normalized_spatial_activity,
    layer_results,
    'L6',  # Choose a layer
    scaled_bin_centers,
    top_n=20
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec

def create_layer_response_plot(normalized_spatial_activity, reliable_cells, layer_cells, layer_name, bin_centers=None, clim=None):
    """
    Create an enhanced response plot for cells in a specific cortical layer, sorted by their peak locations.
    
    Parameters:
    -----------
    normalized_spatial_activity : numpy.ndarray
        Already normalized activity matrix (n_cells x n_trials x n_spatial_bins)
    reliable_cells : numpy.ndarray
        Boolean array indicating reliable cells
    layer_cells : dict
        Dictionary with indices of cells in each layer
    layer_name : str
        Name of the layer to visualize (e.g., 'L2/3', 'L4', 'L5', 'L6')
    bin_centers : numpy.ndarray, optional
        Centers of spatial bins for x-axis labeling
    clim : tuple or None
        Optional color limits (min, max) to enhance contrast. If None, auto-calculated.
        
    Returns:
    --------
    fig : matplotlib.figure.Figure
        Figure with the enhanced sorted response plot
    sorted_layer_indices : numpy.ndarray
        Indices of reliable cells in the specified layer, sorted by peak location
    """
    # Get dimensions
    n_cells, n_trials, n_bins = normalized_spatial_activity.shape
    
    # Check if the layer exists in the layer_cells dictionary
    if layer_name not in layer_cells:
        raise ValueError(f"Layer '{layer_name}' not found in layer_cells")
    
    # Get reliable cells in the specified layer
    reliable_indices = np.where(reliable_cells)[0]
    layer_indices = layer_cells[layer_name]
    layer_reliable_indices = np.intersect1d(reliable_indices, layer_indices)
    
    if len(layer_reliable_indices) == 0:
        print(f"No reliable cells found in {layer_name}")
        return None, None
    
    print(f"{layer_name}: {len(layer_reliable_indices)} reliable cells (out of {len(layer_indices)} total cells)")
    
    # Extract activity for reliable cells in this layer
    layer_activity = normalized_spatial_activity[layer_reliable_indices]
    
    # Split trials into even and odd
    even_trials = np.arange(0, n_trials, 2)
    odd_trials = np.arange(1, n_trials, 2)
    
    # Calculate average responses for each set of trials
    even_avg = np.mean(layer_activity[:, even_trials, :], axis=1)
    odd_avg = np.mean(layer_activity[:, odd_trials, :], axis=1)
    
    # Enhance contrast in the odd trial data for better peak detection
    enhanced_odd_avg = np.power(odd_avg, 0.5)
    
    # Find peak location for each cell
    peak_locations = np.argmax(enhanced_odd_avg, axis=1)
    
    # Sort the cell indices by their peak locations
    sorted_indices = np.argsort(peak_locations)
    
    # Apply sorting to trials for display
    sorted_even_activity = even_avg[sorted_indices]
    sorted_odd_activity = odd_avg[sorted_indices]
    
    # Normalize each cell's response to 0-1 range for better visualization
    for i in range(len(sorted_indices)):
        # Normalize even trials
        cell_min = np.min(sorted_even_activity[i])
        cell_max = np.max(sorted_even_activity[i])
        if cell_max > cell_min:
            sorted_even_activity[i] = (sorted_even_activity[i] - cell_min) / (cell_max - cell_min)
            
        # Normalize odd trials
        cell_min = np.min(sorted_odd_activity[i])
        cell_max = np.max(sorted_odd_activity[i])
        if cell_max > cell_min:
            sorted_odd_activity[i] = (sorted_odd_activity[i] - cell_min) / (cell_max - cell_min)
    
    # Create the figure with a grid layout
    fig = plt.figure(figsize=(20, 10))
    
    # Create main GridSpec with proper spacing for colorbar
    outer_grid = GridSpec(1, 2, width_ratios=[4, 1], wspace=0.25)
    
    # Create nested GridSpec for the left panel (histogram + heatmap)
    left_grid = GridSpec(2, 1, height_ratios=[1, 4], hspace=0.05)
    left_grid.update(left=outer_grid[0, 0].get_position(fig).bounds[0],
                    right=outer_grid[0, 0].get_position(fig).bounds[0] + 
                          outer_grid[0, 0].get_position(fig).bounds[2],
                    bottom=outer_grid[0, 0].get_position(fig).bounds[1],
                    top=outer_grid[0, 0].get_position(fig).bounds[1] + 
                         outer_grid[0, 0].get_position(fig).bounds[3])
    
    # Create axes
    ax_hist = fig.add_subplot(left_grid[0, 0])
    ax_main = fig.add_subplot(left_grid[1, 0], sharex=ax_hist)
    ax_examples = fig.add_subplot(outer_grid[0, 1])
    
    # Create a vibrant colormap with stronger contrast
    cmap = LinearSegmentedColormap.from_list('EnhancedBlues', 
                                         [(1,1,1), (0.8,0.8,1), (0.4,0.4,0.9), (0,0,0.8), (0,0,0.5)])
    
    # Auto-calculate color limits if not provided
    if clim is None:
        vmin = np.percentile(sorted_odd_activity, 5)
        vmax = np.percentile(sorted_odd_activity, 95)
        if vmax - vmin < 0.1:
            vmin = 0
            vmax = 1
    else:
        vmin, vmax = clim
    
    # Plot the spatial heat map in the main panel
    im = ax_main.imshow(sorted_odd_activity, aspect='auto', cmap=cmap, 
                      interpolation='nearest', vmin=vmin, vmax=vmax)
    
    # Add colorbar to the right of the heatmap but inside the left panel
    # Create a small axes for the colorbar
    cbar_ax = fig.add_axes([left_grid[1, 0].get_position(fig).bounds[0] + 
                           left_grid[1, 0].get_position(fig).bounds[2] + 0.01,
                           left_grid[1, 0].get_position(fig).bounds[1],
                           0.015,
                           left_grid[1, 0].get_position(fig).bounds[3]])
    
    # Add the colorbar
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label('Normalized Activity')
    
    # Add labels and title for main panel
    ax_main.set_xlabel('Position (cm)' if bin_centers is not None else 'Spatial Bin')
    ax_main.set_ylabel('Cell Number (sorted by peak location)')
    
    # If bin_centers is provided, use it for x-axis
    if bin_centers is not None:
        n_ticks = 5
        tick_indices = np.linspace(0, len(bin_centers)-1, n_ticks, dtype=int)
        ax_main.set_xticks(tick_indices)
        ax_main.set_xticklabels([f"{bin_centers[i]:.0f}" for i in tick_indices])
    
    # Add title to the whole figure
    fig.suptitle(f'Spatial Responses in {layer_name} ({len(layer_reliable_indices)} Reliable Cells)', 
                fontsize=16, y=0.98)
    
    # [Rest of plotting code remains the same]
    
    # Plot histogram of peak positions in the top panel
    peak_positions = np.argmax(sorted_odd_activity, axis=1)
    ax_hist.hist(peak_positions, bins=n_bins//4, color='skyblue', alpha=0.7)
    ax_hist.set_ylabel('Count')
    ax_hist.set_title('Distribution of Peak Positions')
    # Hide x-labels for histogram
    plt.setp(ax_hist.get_xticklabels(), visible=False)
    
    # Make the main plot and histogram perfectly aligned
    # Ensure tighter limits for histogram to match heatmap
    ax_hist.set_xlim(ax_main.get_xlim())
    
    # Select a few example cells to show in the right panel
    n_examples = min(5, len(sorted_indices))
    # Choose examples evenly spaced through the sorted array
    example_indices = np.linspace(0, len(sorted_indices)-1, n_examples, dtype=int)
    
     # Clear the right panel axis for custom plotting
    ax_examples.clear()
    
    # Select a few example cells to show in the right panel
    n_examples = min(5, len(sorted_indices))
    # Choose examples evenly spaced through the sorted array
    example_indices = np.linspace(0, len(sorted_indices)-1, n_examples, dtype=int)
    
    # MODIFIED: Better spacing parameters to avoid first cell being cropped
    top_margin = 0.2     # Increased top margin to prevent cropping
    bottom_margin = 0.0  # Small bottom margin
    usable_height = 1.0 - top_margin - bottom_margin
    
    # Plot each example cell
    for i, idx in enumerate(example_indices):
        # Get the cell's response
        cell_response = sorted_odd_activity[idx]
        
        # Modified vertical positioning to ensure first cell isn't cropped
        # Space cells evenly across the usable height
        y_pos = 1.0 - top_margin - (i * usable_height / n_examples)
        
        # Scale the response to fit in the panel
        y_height = (usable_height / n_examples) * 0.75  # Reduced height to prevent overlap
        scaled_response = cell_response * y_height
        
        # Horizontal positions (assuming bin centers start at 0)
        x_pos = np.arange(len(cell_response))
        if bin_centers is not None:
            x_pos = bin_centers
        
        # Plot the response with the correct vertical offset
        ax_examples.plot(x_pos, scaled_response + y_pos, 'b-', linewidth=1.5)
        
        # Fill below the curve
        ax_examples.fill_between(x_pos, y_pos, scaled_response + y_pos, alpha=0.3, color='blue')
        
        # Mark the peak
        peak_idx = np.argmax(cell_response)
        peak_x = x_pos[peak_idx] if bin_centers is not None else peak_idx
        ax_examples.scatter(peak_x, cell_response[peak_idx] * y_height + y_pos, 
                          color='red', s=40, zorder=10)
        
        # Add cell number - position slightly lower to avoid title overlap
        text_y_pos = y_pos + y_height*0.7  # Moved down slightly
        ax_examples.text(0.05, text_y_pos, f"Cell {layer_reliable_indices[sorted_indices[idx]]}", 
                       fontsize=8, ha='left', va='top')
    
    # Set up right panel axis
    ax_examples.set_ylim(0.0, 1.0)
    ax_examples.set_title('Example Cell Responses', pad=10)  # Added padding to title
    ax_examples.set_xlabel('Position (cm)' if bin_centers is not None else 'Spatial Bin')
    ax_examples.set_yticks([])  # Hide y-axis ticks
    
    # If bin_centers is provided, use it for x-axis
    if bin_centers is not None:
        # Get tick positions at regular intervals
        n_ticks = 3
        tick_indices = np.linspace(0, len(bin_centers)-1, n_ticks, dtype=int)
        ax_examples.set_xticks([bin_centers[i] for i in tick_indices])
        ax_examples.set_xticklabels([f"{bin_centers[i]:.0f}" for i in tick_indices])
    
    # Get sorted indices of layer cells
    sorted_layer_indices = layer_reliable_indices[sorted_indices]
    
    return fig, sorted_layer_indices   

def plot_all_layers_response(normalized_spatial_activity, reliable_cells, layer_cells, bin_centers=None, clim=None):
    """
    Create response plots for all cortical layers and return them in a dictionary.
    
    Parameters:
    -----------
    normalized_spatial_activity : numpy.ndarray
        Already normalized activity matrix (n_cells x n_trials x n_spatial_bins)
    reliable_cells : numpy.ndarray
        Boolean array indicating reliable cells
    layer_cells : dict
        Dictionary with indices of cells in each layer
    bin_centers : numpy.ndarray, optional
        Centers of spatial bins for x-axis labeling
    clim : tuple or None
        Optional color limits (min, max) to enhance contrast. If None, auto-calculated.
        
    Returns:
    --------
    layer_figs : dict
        Dictionary with layer names as keys and (figure, sorted_indices) tuples as values
    """
    layer_figs = {}
    
    # Process each layer
    for layer_name in layer_cells.keys():
        # Skip empty layers
        if len(layer_cells[layer_name]) == 0:
            print(f"No cells found in {layer_name}")
            continue
            
        # Create response plot for this layer
        fig, sorted_indices = create_layer_response_plot(
            normalized_spatial_activity, 
            reliable_cells, 
            layer_cells, 
            layer_name, 
            bin_centers=bin_centers,
            clim=clim
        )
        
        # Store the figure and sorted indices
        if fig is not None:
            layer_figs[layer_name] = (fig, sorted_indices)
    
    return layer_figs

def compare_layers_response(normalized_spatial_activity, reliable_cells, layer_cells, bin_centers=None, clim=None):
    """
    Create a comparative plot showing the response patterns across all cortical layers.
    
    Parameters:
    -----------
    normalized_spatial_activity : numpy.ndarray
        Already normalized activity matrix (n_cells x n_trials x n_spatial_bins)
    reliable_cells : numpy.ndarray
        Boolean array indicating reliable cells
    layer_cells : dict
        Dictionary with indices of cells in each layer
    bin_centers : numpy.ndarray, optional
        Centers of spatial bins for x-axis labeling
    clim : tuple or None
        Optional color limits (min, max) to enhance contrast. If None, auto-calculated.
        
    Returns:
    --------
    fig : matplotlib.figure.Figure
        Figure with comparative response plots for all layers
    """
    # Count number of layers with reliable cells
    valid_layers = []
    for layer_name, layer_indices in layer_cells.items():
        reliable_layer_indices = np.intersect1d(np.where(reliable_cells)[0], layer_indices)
        if len(reliable_layer_indices) > 0:
            valid_layers.append(layer_name)
    
    if len(valid_layers) == 0:
        print("No reliable cells found in any layer")
        return None
    
    # Create figure with one row per layer
    fig = plt.figure(figsize=(14, 4*len(valid_layers)))
    gs = GridSpec(len(valid_layers), 1)
    
    # Create a vibrant colormap with stronger contrast
    cmap = LinearSegmentedColormap.from_list('EnhancedBlues', 
                                          [(1,1,1), (0.8,0.8,1), (0.4,0.4,0.9), (0,0,0.8), (0,0,0.5)])
    
    # Process each layer
    for i, layer_name in enumerate(valid_layers):
        # Get reliable cells in this layer
        layer_indices = layer_cells[layer_name]
        layer_reliable_indices = np.intersect1d(np.where(reliable_cells)[0], layer_indices)
        
        if len(layer_reliable_indices) == 0:
            continue
        
        # Extract activity for reliable cells in this layer
        layer_activity = normalized_spatial_activity[layer_reliable_indices]
        
        # Calculate average across all trials
        trial_avg = np.mean(layer_activity, axis=1)
        
        # Sort cells by peak position
        peak_positions = np.argmax(trial_avg, axis=1)
        sorted_indices = np.argsort(peak_positions)
        sorted_activity = trial_avg[sorted_indices]
        
        # Normalize each cell's response to 0-1 range
        for j in range(len(sorted_indices)):
            cell_min = np.min(sorted_activity[j])
            cell_max = np.max(sorted_activity[j])
            if cell_max > cell_min:
                sorted_activity[j] = (sorted_activity[j] - cell_min) / (cell_max - cell_min)
        
        # Create axis for this layer
        ax = fig.add_subplot(gs[i])
        
        # Auto-calculate color limits if not provided
        if clim is None:
            vmin = np.percentile(sorted_activity, 5)
            vmax = np.percentile(sorted_activity, 95)
            if vmax - vmin < 0.1:
                vmin = 0
                vmax = 1
        else:
            vmin, vmax = clim
        
        # Plot the response heatmap
        im = ax.imshow(sorted_activity, aspect='auto', cmap=cmap,
                      interpolation='nearest', vmin=vmin, vmax=vmax)
        
        # Add colorbar
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label('Normalized Activity')
        
        # Add labels
        ax.set_title(f'{layer_name}: {len(layer_reliable_indices)} Reliable Cells')
        
        # Only add x-axis label for the bottom plot
        if i == len(valid_layers) - 1:
            ax.set_xlabel('Position (cm)' if bin_centers is not None else 'Spatial Bin')
        
        ax.set_ylabel('Cell Number\n(sorted by peak)')
        
        # If bin_centers is provided, use it for x-axis
        if bin_centers is not None:
            # Get tick positions at regular intervals
            n_ticks = 5
            tick_indices = np.linspace(0, len(bin_centers)-1, n_ticks, dtype=int)
            ax.set_xticks(tick_indices)
            ax.set_xticklabels([f"{bin_centers[i]:.0f}" for i in tick_indices])
    
    fig.suptitle('Comparison of Spatial Response Patterns Across Cortical Layers', fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.97])  # Adjust layout to make room for the title
    
    return fig


fig = plot_all_layers_response(
    normalized_spatial_activity,
    reliable_valid_cells,
    layer_cells,
    bin_centers=scaled_bin_centers
)

# For a comparative figure showing all layers:
comparison_fig = compare_layers_response(
    normalized_spatial_activity,
    reliable_valid_cells,
    layer_cells,
    bin_centers=scaled_bin_centers
)

**Test for adaptation and detrend, if necessary**

In [ ]:
def analyze_single_lap_adaptation(filtered_spks_laps, filtered_location_laps, reliable_cells, bin_centers, n_valid_laps):
    """
    Analyze how neural responses adapt within a single lap.
    
    Parameters:
    -----------
    filtered_spks_laps : list
        List of spike data for each lap (each array is cells x lap_time_points)
    filtered_location_laps : list
        List of location data for each lap
    reliable_cells : numpy.ndarray
        Boolean array indicating reliable cells
    bin_centers : numpy.ndarray
        Centers of spatial bins
    n_valid_laps : int
        Number of valid laps
    """
    print("\nAnalyzing single-lap adaptation effects...")
    
    # Get cell indices of reliable cells
    reliable_indices = np.where(reliable_cells)[0]
    n_reliable = len(reliable_indices)
    
    # For each reliable cell, analyze raw activity trace during each lap
    # We'll normalize time within each lap to compare laps of different durations
    normalized_lap_traces = []
    
    for lap_idx in range(n_valid_laps):
        lap_spks = filtered_spks_laps[lap_idx]
        lap_time_points = lap_spks.shape[1]
        
        # Create normalized time points (0 to 1) for this lap
        norm_time = np.linspace(0, 1, lap_time_points)
        
        # Extract reliable cells' activity
        reliable_spks = lap_spks[reliable_indices, :]
        
        # Store as tuple (normalized time, spikes)
        normalized_lap_traces.append((norm_time, reliable_spks))
    
    # Interpolate all laps to a common time basis for averaging
    n_time_points = 100  # Use 100 normalized time points
    common_time = np.linspace(0, 1, n_time_points)
    
    # Initialize array for interpolated traces
    interp_traces = np.zeros((n_reliable, n_valid_laps, n_time_points))
    
    # Interpolate each lap's data
    for lap_idx in range(n_valid_laps):
        norm_time, reliable_spks = normalized_lap_traces[lap_idx]
        
        for cell_idx in range(n_reliable):
            # Interpolate this cell's activity to common time basis
            interp_traces[cell_idx, lap_idx] = np.interp(
                common_time, norm_time, reliable_spks[cell_idx])
    
    # Calculate average trace across laps for each cell
    avg_traces = np.mean(interp_traces, axis=1)
    
    # Analyze first half vs second half of normalized time
    first_half_activity = np.mean(avg_traces[:, :n_time_points//2], axis=1)
    second_half_activity = np.mean(avg_traces[:, n_time_points//2:], axis=1)
    
    # Calculate adaptation ratio for each cell (first half / second half)
    adaptation_ratio = first_half_activity / second_half_activity
    
    # T-test comparing first and second half
    t_stat, p_value = stats.ttest_rel(first_half_activity, second_half_activity)
    
    # Plot results
    plt.figure(figsize=(15, 10))
    
    # Plot 1: Average normalized time course across all cells
    plt.subplot(2, 2, 1)
    mean_trace = np.mean(avg_traces, axis=0)
    sem_trace = stats.sem(avg_traces, axis=0)
    
    plt.plot(common_time, mean_trace, 'b-', linewidth=2)
    plt.fill_between(common_time, mean_trace - sem_trace, mean_trace + sem_trace,
                    alpha=0.3, color='b')
    
    plt.axvline(0.5, color='k', linestyle='--', alpha=0.5)
    plt.title('Average Normalized Time Course (All Cells)')
    plt.xlabel('Normalized Time Within Lap (0-1)')
    plt.ylabel('Activity')
    
    # Plot 2: Distribution of adaptation ratios
    plt.subplot(2, 2, 2)
    # Cap extreme ratios for better visualization
    capped_ratios = np.clip(adaptation_ratio, 0, 5)
    plt.hist(capped_ratios, bins=30, color='skyblue', edgecolor='black')
    plt.axvline(1, color='r', linestyle='--', alpha=0.5)
    plt.title('Distribution of Adaptation Ratios')
    plt.xlabel('First Half / Second Half Activity')
    plt.ylabel('Count')
    
    # Add mean ratio and p-value
    mean_ratio = np.mean(adaptation_ratio)
    plt.text(0.05, 0.95, f'Mean Ratio: {mean_ratio:.2f}\np-value: {p_value:.4f}',
            transform=plt.gca().transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Plot 3: Examples of cells with different adaptation patterns
    plt.subplot(2, 2, 3)
    
    # Find cells with strong, weak, and no adaptation
    sorted_idx = np.argsort(adaptation_ratio)
    no_adapt_idx = sorted_idx[np.abs(adaptation_ratio[sorted_idx] - 1).argmin()]
    strong_adapt_idx = sorted_idx[-1]  # Highest ratio
    weak_adapt_idx = sorted_idx[0]  # Lowest ratio
    
    # Normalize traces for better visibility
    norm_strong = avg_traces[strong_adapt_idx] / np.max(avg_traces[strong_adapt_idx])
    norm_no = avg_traces[no_adapt_idx] / np.max(avg_traces[no_adapt_idx])
    norm_weak = avg_traces[weak_adapt_idx] / np.max(avg_traces[weak_adapt_idx])
    
    plt.plot(common_time, norm_strong, 'r-', linewidth=2, 
            label=f'Strong Adaptation (ratio={adaptation_ratio[strong_adapt_idx]:.2f})')
    plt.plot(common_time, norm_no, 'g-', linewidth=2, 
            label=f'No Adaptation (ratio={adaptation_ratio[no_adapt_idx]:.2f})')
    plt.plot(common_time, norm_weak, 'b-', linewidth=2, 
            label=f'Weak/Reverse Adaptation (ratio={adaptation_ratio[weak_adapt_idx]:.2f})')
    
    plt.axvline(0.5, color='k', linestyle='--', alpha=0.5)
    plt.title('Example Cells with Different Adaptation Patterns')
    plt.xlabel('Normalized Time Within Lap (0-1)')
    plt.ylabel('Normalized Activity')
    plt.legend()
    
    # Plot 4: Compare adaptation ratio with spatial preference
    plt.subplot(2, 2, 4)
    
    # We need spatial preference data, which we calculate from spatial_activity
    halfway_point = np.median(bin_centers)
    first_half_idx = bin_centers < halfway_point
    second_half_idx = bin_centers >= halfway_point
    
    spatial_preference_ratios = []
    
    # Calculate spatial preference for each reliable cell
    for i, cell_idx in enumerate(reliable_indices):
        activity = trial_averaged_activity[cell_idx]
        first_half = np.mean(activity[first_half_idx])
        second_half = np.mean(activity[second_half_idx])
        
        if second_half > 0:
            ratio = first_half / second_half
        else:
            ratio = 1 if first_half == 0 else 5  # Cap for visualization
            
        spatial_preference_ratios.append(ratio)
    
    # Cap extreme values for better visualization
    spatial_preference_ratios = np.clip(np.array(spatial_preference_ratios), 0, 5)
    
    # Scatter plot
    plt.scatter(spatial_preference_ratios, capped_ratios, alpha=0.6)
    plt.axhline(1, color='r', linestyle='--', alpha=0.5)
    plt.axvline(1, color='r', linestyle='--', alpha=0.5)
    
    # Calculate correlation
    valid_idx = np.isfinite(spatial_preference_ratios) & np.isfinite(capped_ratios)
    if np.sum(valid_idx) > 1:
        corr = np.corrcoef(spatial_preference_ratios[valid_idx], capped_ratios[valid_idx])[0, 1]
        plt.text(0.05, 0.95, f'Correlation: {corr:.3f}',
                transform=plt.gca().transAxes, fontsize=10,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.title('Temporal Adaptation vs Spatial Preference')
    plt.xlabel('Spatial Preference (First/Second Half)')
    plt.ylabel('Temporal Adaptation (First/Second Half)')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"Temporal adaptation analysis:")
    print(f"Mean first half / second half ratio: {mean_ratio:.2f}")
    print(f"Paired t-test: t={t_stat:.3f}, p={p_value:.4f}")
    
    if p_value < 0.05:
        direction = "higher" if mean_ratio > 1 else "lower"
        print(f"Activity in the first half of each lap is significantly {direction} than in the second half")
    else:
        print("No significant temporal adaptation within laps")
    
    if 'corr' in locals():
        print(f"Correlation between spatial preference and temporal adaptation: {corr:.3f}")
        
    return avg_traces, adaptation_ratio, interp_traces

def analyze_cross_lap_adaptation(filtered_spks_laps, filtered_location_laps, reliable_cells, n_valid_laps):
    """
    Analyze how neural responses adapt across multiple laps.
    
    Parameters:
    -----------
    filtered_spks_laps : list
        List of spike data for each lap (each array is cells x lap_time_points)
    filtered_location_laps : list
        List of location data for each lap
    reliable_cells : numpy.ndarray
        Boolean array indicating reliable cells
    n_valid_laps : int
        Number of valid laps
    """
    print("\nAnalyzing adaptation across multiple laps...")
    
    # Get reliable cell indices
    reliable_indices = np.where(reliable_cells)[0]
    n_reliable = len(reliable_indices)
    
    # Calculate mean activity per lap for each reliable cell
    mean_lap_activity = np.zeros((n_reliable, n_valid_laps))
    
    for lap_idx in range(n_valid_laps):
        lap_spks = filtered_spks_laps[lap_idx]
        
        # Extract reliable cells' activity
        reliable_spks = lap_spks[reliable_indices, :]
        
        # Calculate mean activity for this lap
        mean_lap_activity[:, lap_idx] = np.mean(reliable_spks, axis=1)
    
    # Normalize for each cell to compare relative changes
    norm_lap_activity = np.zeros_like(mean_lap_activity)
    
    for cell_idx in range(n_reliable):
        cell_activity = mean_lap_activity[cell_idx]
        
        # Normalize by first lap activity
        if cell_activity[0] > 0:
            norm_lap_activity[cell_idx] = cell_activity / cell_activity[0]
        else:
            # If first lap is zero, normalize by max
            max_val = np.max(cell_activity)
            if max_val > 0:
                norm_lap_activity[cell_idx] = cell_activity / max_val
            else:
                norm_lap_activity[cell_idx] = np.ones_like(cell_activity)
    
    # Calculate mean and SEM across cells
    mean_norm_activity = np.mean(norm_lap_activity, axis=0)
    sem_norm_activity = stats.sem(norm_lap_activity, axis=0)
    
    # Test for significant trend across laps
    lap_indices = np.arange(1, n_valid_laps + 1)
    
    # Use Spearman rank correlation to test for monotonic trend
    overall_trend = []
    
    for cell_idx in range(n_reliable):
        rho, p_value = stats.spearmanr(lap_indices, norm_lap_activity[cell_idx])
        overall_trend.append(rho)
    
    # Plot results
    plt.figure(figsize=(15, 10))
    
    # Plot 1: Average normalized activity across laps
    plt.subplot(2, 2, 1)
    plt.errorbar(lap_indices, mean_norm_activity, yerr=sem_norm_activity,
                fmt='o-', capsize=5, color='blue', linewidth=2)
    
    plt.axhline(1, color='r', linestyle='--', alpha=0.5)
    plt.title('Average Normalized Activity Across Laps')
    plt.xlabel('Lap Number')
    plt.ylabel('Activity (Normalized to First Lap)')
    plt.xticks(lap_indices)
    plt.grid(True, alpha=0.3)
    
    # Plot 2: Distribution of adaptation trends
    plt.subplot(2, 2, 2)
    plt.hist(overall_trend, bins=20, color='skyblue', edgecolor='black')
    plt.axvline(0, color='r', linestyle='--', alpha=0.5)
    plt.title('Distribution of Adaptation Trends')
    plt.xlabel('Spearman Correlation with Lap Number')
    plt.ylabel('Count')
    
    # Add mean trend and significance
    mean_trend = np.mean(overall_trend)
    t_stat, p_value = stats.ttest_1samp(overall_trend, 0)
    
    plt.text(0.05, 0.95, f'Mean Trend: {mean_trend:.3f}\np-value: {p_value:.4f}',
            transform=plt.gca().transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Plot 3: Example cells with different adaptation patterns
    plt.subplot(2, 2, 3)
    
    # Find cells with strong positive, strong negative, and no adaptation
    sorted_idx = np.argsort(overall_trend)
    no_adapt_idx = sorted_idx[np.abs(np.array(overall_trend)).argmin()]
    positive_idx = sorted_idx[-1]  # Most positive trend
    negative_idx = sorted_idx[0]  # Most negative trend
    
    plt.plot(lap_indices, norm_lap_activity[positive_idx], 'g-o', linewidth=2, 
            label=f'Increasing (rho={overall_trend[positive_idx]:.2f})')
    plt.plot(lap_indices, norm_lap_activity[no_adapt_idx], 'b-o', linewidth=2, 
            label=f'Stable (rho={overall_trend[no_adapt_idx]:.2f})')
    plt.plot(lap_indices, norm_lap_activity[negative_idx], 'r-o', linewidth=2, 
            label=f'Decreasing (rho={overall_trend[negative_idx]:.2f})')
    
    plt.axhline(1, color='k', linestyle='--', alpha=0.5)
    plt.title('Example Cells with Different Adaptation Patterns')
    plt.xlabel('Lap Number')
    plt.ylabel('Normalized Activity')
    plt.xticks(lap_indices)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 4: Split cells by first-half vs second-half preference
    plt.subplot(2, 2, 4)
    
    # We need spatial preference data to calculate this
    halfway_point = np.median(bin_centers)
    first_half_idx = bin_centers < halfway_point
    second_half_idx = bin_centers >= halfway_point
    
    first_half_preferring = []
    second_half_preferring = []
    
    # Identify cells with first-half vs second-half preference
    for i, cell_idx in enumerate(reliable_indices):
        activity = trial_averaged_activity[cell_idx]
        first_half = np.mean(activity[first_half_idx])
        second_half = np.mean(activity[second_half_idx])
        
        if first_half > second_half:
            first_half_preferring.append(i)
        else:
            second_half_preferring.append(i)
    
    # Calculate mean activity for each group
    mean_first_pref = np.mean(norm_lap_activity[first_half_preferring], axis=0) if first_half_preferring else None
    sem_first_pref = stats.sem(norm_lap_activity[first_half_preferring], axis=0) if first_half_preferring else None
    
    mean_second_pref = np.mean(norm_lap_activity[second_half_preferring], axis=0) if second_half_preferring else None
    sem_second_pref = stats.sem(norm_lap_activity[second_half_preferring], axis=0) if second_half_preferring else None
    
    # Plot the two groups
    if mean_first_pref is not None:
        plt.errorbar(lap_indices, mean_first_pref, yerr=sem_first_pref,
                    fmt='o-', capsize=5, color='blue', linewidth=2,
                    label=f'First-half preferring (n={len(first_half_preferring)})')
    
    if mean_second_pref is not None:
        plt.errorbar(lap_indices, mean_second_pref, yerr=sem_second_pref,
                    fmt='o-', capsize=5, color='red', linewidth=2,
                    label=f'Second-half preferring (n={len(second_half_preferring)})')
    
    plt.axhline(1, color='k', linestyle='--', alpha=0.5)
    plt.title('Adaptation by Spatial Preference')
    plt.xlabel('Lap Number')
    plt.ylabel('Normalized Activity')
    plt.xticks(lap_indices)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"Cross-lap adaptation analysis:")
    print(f"Mean adaptation trend (Spearman rho): {mean_trend:.3f}")
    print(f"One-sample t-test against zero: t={t_stat:.3f}, p={p_value:.4f}")
    
    if p_value < 0.05:
        direction = "increase" if mean_trend > 0 else "decrease"
        print(f"Significant {direction} in activity across laps")
    else:
        print("No significant adaptation trend across laps")
    
    if mean_first_pref is not None and mean_second_pref is not None:
        # Test for difference in adaptation between first-half and second-half preferring cells
        first_trends = [overall_trend[i] for i in first_half_preferring]
        second_trends = [overall_trend[i] for i in second_half_preferring]
        
        if first_trends and second_trends:
            t_stat, p_value = stats.ttest_ind(first_trends, second_trends, equal_var=False)
            print(f"\nComparison of adaptation between spatial preference groups:")
            print(f"First-half preferring cells mean trend: {np.mean(first_trends):.3f}")
            print(f"Second-half preferring cells mean trend: {np.mean(second_trends):.3f}")
            print(f"t-test: t={t_stat:.3f}, p={p_value:.4f}")
            
            if p_value < 0.05:
                stronger = "first-half" if np.mean(first_trends) < np.mean(second_trends) else "second-half"
                print(f"Significant difference in adaptation trends: {stronger} preferring cells show stronger adaptation")
    
    return overall_trend, norm_lap_activity

# Run adaptation analysis
avg_traces, within_lap_adaptation, normalized_traces = analyze_single_lap_adaptation(
    filtered_spks_laps, filtered_location_laps, reliable_cells, bin_centers, n_valid_laps)

cross_lap_adaptation, normalized_lap_activity = analyze_cross_lap_adaptation(
    filtered_spks_laps, filtered_location_laps, reliable_cells, n_valid_laps)

In [ ]:
def analyze_adaptation_by_layer(normalized_traces, adaptation_ratio, layer_cells, reliable_cells, common_time):
    """
    Analyze adaptation properties by cortical layer.
    
    Parameters:
    -----------
    normalized_traces : numpy.ndarray
        Normalized activity traces (cells x time points)
    adaptation_ratio : numpy.ndarray
        First half / second half activity ratio for each cell
    layer_cells : dict
        Dictionary with indices of cells in each layer
    reliable_cells : numpy.ndarray
        Boolean array indicating reliable cells
    common_time : numpy.ndarray
        Common time basis for normalized traces
    """
    print("\nAnalyzing adaptation by cortical layer...")
    
    # Get indices of reliable cells
    reliable_indices = np.where(reliable_cells)[0]
    
    # Collect layer-specific adaptation data
    layer_adaptation = {}
    layer_traces = {}
    
    # For each layer, extract adaptation ratios and traces for reliable cells
    for layer_name, cell_indices in layer_cells.items():
        # Find reliable cells in this layer
        reliable_layer_cells = np.intersect1d(reliable_indices, cell_indices)
        
        if len(reliable_layer_cells) > 0:
            # Find positions of these cells in the reliable_indices array
            positions = []
            for idx in reliable_layer_cells:
                pos = np.where(reliable_indices == idx)[0]
                if len(pos) > 0:
                    positions.append(pos[0])
            
            # Extract adaptation ratios and traces for this layer
            layer_adaptation[layer_name] = adaptation_ratio[positions]
            layer_traces[layer_name] = normalized_traces[positions]
    
    # Create a figure for layer-specific analysis
    plt.figure(figsize=(15, 12))
    
    # 1. Box plot of adaptation ratios by layer
    plt.subplot(2, 2, 1)
    
    boxplot_data = []
    layer_names = []
    
    for layer_name, ratios in layer_adaptation.items():
        # Cap extreme values for better visualization
        capped_ratios = np.clip(ratios, 0, 5)
        boxplot_data.append(capped_ratios)
        layer_names.append(f"{layer_name} (n={len(ratios)})")
    
    plt.boxplot(boxplot_data, labels=layer_names)
    plt.axhline(1, color='r', linestyle='--', alpha=0.5)
    plt.title('Adaptation Ratio by Layer')
    plt.ylabel('First Half / Second Half Activity Ratio')
    plt.xticks(rotation=45)
    
    # 2. Bar plot of mean adaptation ratios by layer
    plt.subplot(2, 2, 2)
    
    means = [np.mean(ratios) for ratios in boxplot_data]
    sems = [stats.sem(ratios) for ratios in boxplot_data]
    
    plt.bar(range(len(means)), means, yerr=sems, capsize=10, color='skyblue')
    plt.axhline(1, color='r', linestyle='--', alpha=0.5)
    plt.xticks(range(len(means)), layer_names, rotation=45)
    plt.title('Mean Adaptation Ratio by Layer')
    plt.ylabel('Mean First/Second Half Ratio')
    
    # 3. Average temporal profiles by layer
    plt.subplot(2, 2, 3)
    
    halfway_idx = len(common_time) // 2
    colors = ['blue', 'orange', 'green', 'red']
    
    for i, (layer_name, traces) in enumerate(layer_traces.items()):
        # Calculate mean and SEM
        mean_trace = np.mean(traces, axis=0)
        sem_trace = stats.sem(traces, axis=0)
        
        # Plot with confidence interval
        plt.plot(common_time, mean_trace, 
                color=colors[i % len(colors)], 
                linewidth=2, 
                label=f"{layer_name} (n={len(traces)})")
        
        plt.fill_between(common_time, 
                        mean_trace - sem_trace, 
                        mean_trace + sem_trace,
                        alpha=0.2, 
                        color=colors[i % len(colors)])
    
    plt.axvline(0.5, color='k', linestyle='--', alpha=0.5)
    plt.title('Average Normalized Time Course by Layer')
    plt.xlabel('Normalized Time Within Lap (0-1)')
    plt.ylabel('Activity')
    plt.legend()
    
    # 4. Adaptation magnitude comparison (normalized)
    plt.subplot(2, 2, 4)
    
    # For each layer, calculate first and second half activity
    first_half_activity = {}
    second_half_activity = {}
    
    for layer_name, traces in layer_traces.items():
        first_half_activity[layer_name] = np.mean(traces[:, :halfway_idx], axis=1)
        second_half_activity[layer_name] = np.mean(traces[:, halfway_idx:], axis=1)
    
    # Create grouped bar chart
    x_pos = np.arange(len(layer_names))
    width = 0.35
    
    first_half_means = [np.mean(first_half_activity[layer_name]) for layer_name in layer_adaptation.keys()]
    first_half_sems = [stats.sem(first_half_activity[layer_name]) for layer_name in layer_adaptation.keys()]
    
    second_half_means = [np.mean(second_half_activity[layer_name]) for layer_name in layer_adaptation.keys()]
    second_half_sems = [stats.sem(second_half_activity[layer_name]) for layer_name in layer_adaptation.keys()]
    
    plt.bar(x_pos - width/2, first_half_means, width, 
           color='lightgreen', label='First Half', yerr=first_half_sems, capsize=5)
    plt.bar(x_pos + width/2, second_half_means, width, 
           color='lightcoral', label='Second Half', yerr=second_half_sems, capsize=5)
    
    plt.xticks(x_pos, [name.split(' ')[0] for name in layer_names], rotation=45)
    plt.title('First vs Second Half Activity by Layer')
    plt.ylabel('Mean Activity')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Statistical tests for each layer
    print("\nStatistical analysis by layer:")
    
    for layer_name, ratios in layer_adaptation.items():
        mean_ratio = np.mean(ratios)
        median_ratio = np.median(ratios)
        
        print(f"\n{layer_name}:")
        print(f"  Mean adaptation ratio: {mean_ratio:.2f}")
        print(f"  Median adaptation ratio: {median_ratio:.2f}")
        
        # T-test comparing first and second half
        first = first_half_activity[layer_name]
        second = second_half_activity[layer_name]
        t_stat, p_value = stats.ttest_rel(first, second)
        
        print(f"  Paired t-test (first vs second half): t={t_stat:.3f}, p={p_value:.4f}")
        
        if p_value < 0.05:
            direction = "higher" if np.mean(first) > np.mean(second) else "lower"
            print(f"  Activity in first half is significantly {direction} than second half")
        else:
            print("  No significant difference between first and second half")
            
        # One-sample t-test comparing ratio to 1
        t_stat, p_value = stats.ttest_1samp(ratios, 1)
        print(f"  Adaptation ratio t-test against 1: t={t_stat:.3f}, p={p_value:.4f}")
        
        if p_value < 0.05:
            direction = "adaptation" if mean_ratio > 1 else "facilitation"
            print(f"  Significant {direction} effect")
    
    # Compare adaptation between layers
    if len(layer_adaptation) > 1:
        print("\nBetween-layer comparisons:")
        
        layer_keys = list(layer_adaptation.keys())
        
        for i, layer1 in enumerate(layer_keys):
            for j, layer2 in enumerate(layer_keys[i+1:], i+1):
                ratios1 = layer_adaptation[layer1]
                ratios2 = layer_adaptation[layer2]
                
                t_stat, p_value = stats.ttest_ind(ratios1, ratios2, equal_var=False)
                print(f"  {layer1} vs {layer2}: t={t_stat:.3f}, p={p_value:.4f}")
                
                if p_value < 0.05:
                    stronger = layer1 if np.mean(ratios1) > np.mean(ratios2) else layer2
                    print(f"    {stronger} shows significantly stronger adaptation")
    
    return layer_adaptation, layer_traces

# Get the common time axis from the previous adaptation analysis
common_time = np.linspace(0, 1, 100)  # Assuming 100 normalized time points as in the previous function

# Run layer-specific adaptation analysis
layer_adaptation, layer_traces = analyze_adaptation_by_layer(
    avg_traces,  # From analyze_single_lap_adaptation
    within_lap_adaptation,  # From analyze_single_lap_adaptation
    layer_cells,
    reliable_cells,
    common_time
)

In [ ]:
# Then run the adaptation-corrected analysis:
detrended_activity, detrended_smi_results, layer_results = DA.run_adaptation_corrected_smi_analysis(
    normalized_spatial_activity, scaled_bin_centers, reliable_valid_cells, layer_cells, segment_distance=segment_distance, exclude_boundary_cm=exclude_boundary_cm)


In [ ]:
# save SMI results to a file, in the filepath directory
save_path = os.path.join(filepath, twoP_filename + '_layer_smi_results.npy')
print(save_path)
np.save(save_path, layer_results)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.stats import ttest_rel, ttest_ind
import warnings
warnings.filterwarnings('ignore')

# Day 1 SMI data
day1_l23 = np.array([ 0.82529719,  0.73281749,  0.86077187,  0.99571221,  0.96941761,  0.61291271,
    0.46293179,  0.34408708,  0.28865708,  0.66325987,  0.30983131,  0.21669633,
 -0.12903455,  0.4497391,   0.70517466,  0.39900934,  0.83559186, -0.06880769,
    0.0836865,   0.56225291,  0.67231174,  0.52935668,  0.72349828,  0.70669006,
    0.21286131,  0.49643675,  0.36459608,  0.55036293, -0.00798494,  0.48540244,
    0.73782254,  0.41783194,  0.52125883,  0.05232466,  0.34085945,  0.40661808,
 -0.04126762,  0.2804813,   0.47061318,  0.49589314, -0.07261818,  0.51092486,
    0.3805408,   0.16711272,  0.08038369,  0.18829303,  0.68563406,  0.3174372,
    0.43277401,  0.40241296,  0.29728722, -0.11551096,  0.167593,    0.51306687,
    0.68765312,  0.42569777,  0.40746337,  0.09383921,  0.30425591,  0.43707023,
    0.4268263,   0.44422134,  0.06188957,  0.62770368,  0.43298816,  0.60018398,
    0.32032529,  0.21640577, -0.11764577,  0.69262949,  0.51204119,  0.34922806,
    0.47885855,  0.27997233,  0.4841043,   0.76585106,  0.46967544,  0.52330068,
    0.41634436,  0.1205718,   0.39145186,  0.23940121,  0.56755614,  0.34040272,
    0.50059948,  0.11620886,  0.47193006,  0.09531832,  0.4501576,   0.40787645,
    0.40181453,  0.52223698,  0.14357124,  0.27754686,  0.21235932,  0.26643958,
    0.31619361,  0.77612323,  0.38741992,  0.70657563,  0.48364064,  0.24341777,
    0.031687,    0.54822211,  0.49144428,  0.43415059,  0.74034474,  0.7337,    ]
)

day1_l4 = np.array([ 0.44314938,  0.91664957,  0.35355798, -0.21718516,  0.72564719, -0.09545941,
    0.66984513,  0.3239312,   1.,          0.77273648,  0.43521865,  0.38868169,
    0.77088767,  0.53595704,  1.,          0.64575541,  1.,          0.7826781,
    1.,         -0.09364543,  0.26256178,  0.9999986,   0.57100837,  0.79906823,
    0.38293053,  1.,          0.72956929,  0.5807233,   0.23733921,  0.74977347,
    0.44776227,  0.49510573,  0.33193234,  0.77480541,  0.33972502,  1.,
 -0.05009669,  0.24202147,  0.33745405,  1.,          1.,          0.15085237,
    0.92693911, -0.0035752,   1.,          0.23066396,  1.,          0.497302,
    0.58327516,  0.69675972,  0.46401847,  0.24741092,  0.39180001,  0.20795034,
    0.62619618,  0.88097129,  0.58779752,  0.47210011,  0.2842005,   0.21734442,
    0.74699268,  0.40928308,  0.70811855,  0.38642053,  0.34886864,  0.11238113,
    0.44674945,  0.45279213,  0.37661476,  0.74491164,  1.,          0.25947085,
    0.02877208,  0.30345844,  0.24903121,  0.5071734,   0.52988257,  0.28113293,
    0.21884144,  0.41249701, -0.01856179,  0.75224445,  0.45243994,  0.57708758,
    0.27773602,  0.23224322,  0.37114385,  0.57550581,  0.37507386,  0.3575403,
    0.38957841,  0.58030014,  0.472205,   -0.01835114,  0.16542085,  1.,
    0.56038788,  0.86837214,  0.27178649, -0.0802393,   0.46202189,  0.45113184,
    0.17084368,  0.50145528,  0.4216788,   0.26441465,  0.51870494, ]
)

day1_l5 = np.array([ 0.86127318,  0.53552362,  0.04773056,  0.22800601,  0.0203303,   0.47142667,
 -0.15959655,  0.58315377,  0.41284852,  0.15540964,  0.26228842,  0.94550066,
    0.35030589,  0.22623846,  1.,          0.71746206,  0.03607839,  0.33047051,
    0.37350745,  0.20080899,  0.020108,    0.44924531,  0.1178781,   0.35540171,
    0.42907896,  0.78503363,  0.03780586,  0.36149928,  0.37889786,  0.37370624,
    0.331409,    0.10110581,  0.46594375,  0.47076382,  0.84208156,  0.73599327,
    0.66494706, -0.11180399,  0.27684869,  0.55448342,  1.,         -0.11475093,
    0.06017602,  0.61524353,  0.24944935,  0.15605383,  0.26526433,  0.19003381,
    0.26983726,  0.14124297,  0.31655794,  0.23081827,  0.56263501,  0.5267595,
    0.6332021,   0.1858246,   0.03983688,  0.24794129,  0.46525873,  0.55627336,
    1.,          0.09995786,  0.32761349,  0.28116229,  0.36587666,  0.36519787,
    0.77342886,  0.37804374,  0.58771546, -0.0047847,   1.,          1.,
    0.52844952,  0.30510885,  0.51336455,  1.,          0.36974769,  0.79284323, ]
)

day1_l6 = np.array([ 0.04039669,  1.,          0.51178406,  0.32069961, -0.13667625,  0.22783702,
 -0.27685683,  0.5823781,   1.,         -0.10441387,  0.63223357,  1.,
    0.66566428,  0.26270882,  1.,         -0.16904466,  0.38898424,  0.39331648,
    1.,          0.68384743,  0.50481658,  0.50763501,  0.99999999,  0.47191937,
    0.55199116,  0.29657983,  0.6168044,  -0.33542208,  0.37848529,  0.85117461,
    0.01601532,  1.,          1.,          0.4828464,   0.534585,    0.39695673,
    0.33480787,  1.,         -0.2137837,   0.24505594,  0.40168342,  0.09048251,
    0.72245132, -0.03988908,  0.14417594, -0.04033172,  0.38272681,  0.30715247,
    0.10151876, -0.09192826,  0.28988712,  0.57734246,  0.30742196,  1.,
 -0.26346496,  0.36276927,  0.55550979,  0.14781419,  0.20668309,  0.23397784,
    0.80916491,  0.5769365,   0.68777859,  0.56000531,  0.72490079,  0.26926735,
    0.35275594, ]
)

# Day 5 SMI data
day5_l23 = np.array([0.90528908, 0.81151141, 0.99263743, 0.68643116, 0.4685717,  0.37525527,
 0.14936086, 0.40208913, 0.53212512, 0.44044256, 0.67346159, 0.69433925,
 0.46773337, 0.71757314, 0.54631702, 0.39212178, 0.35907233, 0.40375517,
 0.64242303, 0.80954168, 0.16717191, 0.76210336, 0.76614555, 0.67491308,
 0.84530478, 0.41987894, 0.80538882, 0.34183523, 0.41906349, 0.22360566,
 0.5238356,  0.34688346, 0.45346505, 0.44722276, 0.6459491,  0.4156503,
 0.2648048,  0.45010644, 0.41124078, 0.46444336, 0.08439037, 0.10916605,
 0.31719065, 0.34996151, 0.57907328, 0.54588333, 0.67131613, 0.46667301,
 0.63247279, 0.43099758, 0.27187202, 0.57382569, 0.37199202, 0.37634032, ]
)

day5_l4 = np.array([ 0.56521248,  0.62334155,  0.31431723,  0.67590121,  0.70398606,  0.44593096,
    0.31868855,  0.37726216,  0.25953473,  0.68197546,  0.36105118,  0.44824576,
    0.27919483,  0.66977429,  0.47490932,  0.63878063, -0.078167,    0.28094579,
0.39888272,  0.43177268,  0.20677855,  0.5574069,   0.39017969, ]
)

day5_l5 = np.array([0.55810065, 0.32834842, 0.99999872, 0.4685954,  0.52446204, 0.8965421,
 0.79629398, 0.18513525, 0.66510588, 0.49965047, 0.61108116, 0.81858586,
 0.71187679, 0.40120585, 0.80662526, 0.42201239, 0.79945521, 0.77411877,
 0.42113405, 0.37934845, 0.68114123, 0.84693755, 0.3645931,  0.80584703,
 0.53164916, 0.45742546, 0.16855525, 0.42729886, 0.45020336, 0.50344522,
 0.5191266,  0.55401177, 0.85318359, ]
)

day5_l6 = np.array([ 0.99295996,  0.64127785,  0.68867622,  0.98728077,  0.68678372,  0.1932694,
    0.82313406,  0.17402183, -0.20329059,  0.5929139,   0.71881721,  0.53098119,
 -0.02327452,  0.68087995,  0.58136071,  0.77493617,  0.70148874,  0.11712496,
    0.27264642,  0.89962909,  0.63486428,  0.93488499,  0.77073773,  0.67850736,
    0.59568943,  0.42353794,  0.87722829,  0.9729653,   0.56230564,  0.26194103,
    0.67065976,  0.53691772, ]
)

def calculate_layer_stats(data, layer_name, day):
    """Calculate descriptive statistics for each layer"""
    return {
        'layer': layer_name,
        'day': day,
        'n': len(data),
        'mean': np.mean(data),
        'sem': stats.sem(data),
        'median': np.median(data),
        'std': np.std(data)
    }

def perform_independent_comparison(day1_data, day5_data, layer_name):
    """Perform independent samples t-test between days for a specific layer"""
    # Use Welch's t-test (unequal variances assumed)
    t_stat, p_value = ttest_ind(day5_data, day1_data, equal_var=False)
    
    # Calculate pooled standard deviation for effect size
    n1, n2 = len(day1_data), len(day5_data)
    pooled_std = np.sqrt(((n1-1)*np.var(day1_data, ddof=1) + (n2-1)*np.var(day5_data, ddof=1)) / (n1+n2-2))
    effect_size = (np.mean(day5_data) - np.mean(day1_data)) / pooled_std
    
    return {
        'layer': layer_name,
        't_stat': t_stat,
        'p_value': p_value,
        'effect_size': effect_size,
        'mean_change': np.mean(day5_data) - np.mean(day1_data),
        'n_day1': n1,
        'n_day5': n2
    }

def create_comparison_dataframe():
    """Create a long-format dataframe for analysis"""
    data = []
    
    # Day 1 data
    for val in day1_l23: data.append({'SMI': val, 'Layer': 'L2/3', 'Day': 1})
    for val in day1_l4: data.append({'SMI': val, 'Layer': 'L4', 'Day': 1})
    for val in day1_l5: data.append({'SMI': val, 'Layer': 'L5', 'Day': 1})
    for val in day1_l6: data.append({'SMI': val, 'Layer': 'L6', 'Day': 1})
    
    # Day 5 data
    for val in day5_l23: data.append({'SMI': val, 'Layer': 'L2/3', 'Day': 5})
    for val in day5_l4: data.append({'SMI': val, 'Layer': 'L4', 'Day': 5})
    for val in day5_l5: data.append({'SMI': val, 'Layer': 'L5', 'Day': 5})
    for val in day5_l6: data.append({'SMI': val, 'Layer': 'L6', 'Day': 5})
    
    return pd.DataFrame(data)

# Calculate descriptive statistics
print("=== DESCRIPTIVE STATISTICS ===\n")
stats_day1 = [
    calculate_layer_stats(day1_l23, 'L2/3', 1),
    calculate_layer_stats(day1_l4, 'L4', 1),
    calculate_layer_stats(day1_l5, 'L5', 1),
    calculate_layer_stats(day1_l6, 'L6', 1)
]

stats_day5 = [
    calculate_layer_stats(day5_l23, 'L2/3', 5),
    calculate_layer_stats(day5_l4, 'L4', 5),
    calculate_layer_stats(day5_l5, 'L5', 5),
    calculate_layer_stats(day5_l6, 'L6', 5)
]

all_stats = stats_day1 + stats_day5
stats_df = pd.DataFrame(all_stats)

print("Layer Statistics by Day:")
print(stats_df.round(3))

# Perform independent samples comparisons between days
print("\n=== INDEPENDENT SAMPLES COMPARISONS BETWEEN DAYS ===\n")
independent_results = [
    perform_independent_comparison(day1_l23, day5_l23, 'L2/3'),
    perform_independent_comparison(day1_l4, day5_l4, 'L4'),
    perform_independent_comparison(day1_l5, day5_l5, 'L5'),
    perform_independent_comparison(day1_l6, day5_l6, 'L6')
]

for result in independent_results:
    print(f"{result['layer']} (Day 1: n={result['n_day1']}, Day 5: n={result['n_day5']}):")
    print(f"  Mean difference (Day 5 - Day 1): {result['mean_change']:.3f}")
    print(f"  t = {result['t_stat']:.3f}, p = {result['p_value']:.4f}")
    print(f"  Effect size (Cohen's d): {result['effect_size']:.3f}")
    
    # Interpret significance and effect
    if result['p_value'] < 0.05:
        direction = "higher" if result['mean_change'] > 0 else "lower"
        print(f"  **SIGNIFICANT: Day 5 SMI {direction.upper()} than Day 1**")
    else:
        print("  No significant difference between days")
    print()

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))
gs = GridSpec(2, 3, figure=fig)
fig.suptitle('SMI Comparison Across Layers Between Day 1 and Day 5', fontsize=16, fontweight='bold')

# Create dataframe for plotting
df = create_comparison_dataframe()
layers = ['L2/3', 'L4', 'L5', 'L6']
colors = ['lightblue', 'lightgreen', 'salmon', 'plum']

# Plot 1: Box plots by layer and day
ax1 = fig.add_subplot(gs[0, 0])
sns.boxplot(data=df, x='Layer', y='SMI', hue='Day', ax=ax1, palette=['lightsteelblue', 'lightcoral'])
ax1.set_title('SMI Distribution by Layer and Day')
ax1.set_ylabel('Spatial Modulation Index')
ax1.legend(title='Day')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Plot 2: Day 5 means with statistical comparison bars
ax2 = fig.add_subplot(gs[0, 1])

# Get Day 5 statistics for each layer
day5_data = [day5_l23, day5_l4, day5_l5, day5_l6]
day5_means = [np.mean(data) for data in day5_data]
day5_sems = [stats.sem(data) for data in day5_data]
layer_names_with_n = [f"{layer}\n(n={len(data)})" for layer, data in zip(layers, day5_data)]

# Create bar plot
bars = ax2.bar(range(len(day5_means)), day5_means, yerr=day5_sems, 
               capsize=10, color=colors, alpha=0.8)

ax2.set_xticks(range(len(day5_means)))
ax2.set_xticklabels(layer_names_with_n)
ax2.axhline(0, color='k', linestyle='--', alpha=0.5)
ax2.set_title('Mean SMI by Layer (Day 5)')
ax2.set_ylabel('Mean SMI ± SEM')

# Add statistical comparison bars between layers (Day 5 only)
max_height = max(day5_means) + max(day5_sems) * 1.5
significant_comparisons = []

# Find significant comparisons between layers on Day 5
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day5_data[i], day5_data[j], equal_var=False)
        if p_value < 0.05:
            significant_comparisons.append((i, j, p_value))

# Sort comparisons for logical ordering
significant_comparisons.sort(key=lambda x: (x[0], x[1] - x[0]))

# Draw comparison bars
bar_spacing = max_height * 0.1
current_height = max_height

for bar_idx, (i, j, p_value) in enumerate(significant_comparisons):
    bar_y = current_height + bar_spacing * bar_idx
    
    # Draw horizontal line
    ax2.plot([i, j], [bar_y, bar_y], 'k-', linewidth=1.5)
    
    # Draw vertical lines
    ax2.plot([i, i], [bar_y, bar_y-bar_spacing*0.3], 'k-', linewidth=1.5)
    ax2.plot([j, j], [bar_y, bar_y-bar_spacing*0.3], 'k-', linewidth=1.5)
    
    # Add significance marker
    if p_value < 0.001:
        sig_text = '***'
    elif p_value < 0.01:
        sig_text = '**'
    elif p_value < 0.05:
        sig_text = '*'
    
    ax2.text((i+j)/2, bar_y+bar_spacing*0.1, sig_text, ha='center', va='bottom', fontsize=12, fontweight='bold')

# Adjust y-axis if we have comparisons
if significant_comparisons:
    ax2.set_ylim(top=current_height + bar_spacing * (len(significant_comparisons) + 1))

# Plot 3: Mean differences between days with statistical comparison
ax3 = fig.add_subplot(gs[0, 2])
mean_changes = [r['mean_change'] for r in independent_results]
p_values = [r['p_value'] for r in independent_results]

# Color bars based on significance
bar_colors = ['red' if p < 0.05 else 'gray' for p in p_values]
bars = ax3.bar(range(len(mean_changes)), mean_changes, color=bar_colors, alpha=0.7)
ax3.set_xticks(range(len(layers)))
ax3.set_xticklabels(layers)
ax3.set_title('Mean SMI Difference (Day 5 - Day 1)')
ax3.set_ylabel('Difference in Mean SMI')
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.8)

# Add significance stars above bars
for i, (bar, p_val) in enumerate(zip(bars, p_values)):
    if p_val < 0.001:
        star = '***'
    elif p_val < 0.01:
        star = '**'
    elif p_val < 0.05:
        star = '*'
    else:
        star = 'ns'
    
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + (0.01 if height > 0 else -0.03),
             star, ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')

# Plot 4: Side-by-side comparison of layer means
ax4 = fig.add_subplot(gs[1, 0])

# Prepare data for side-by-side bars
x = np.arange(len(layers))
width = 0.35

day1_means = [np.mean(data) for data in [day1_l23, day1_l4, day1_l5, day1_l6]]
day5_means = [np.mean(data) for data in [day5_l23, day5_l4, day5_l5, day5_l6]]
day1_sems = [stats.sem(data) for data in [day1_l23, day1_l4, day1_l5, day1_l6]]
day5_sems = [stats.sem(data) for data in [day5_l23, day5_l4, day5_l5, day5_l6]]

bars1 = ax4.bar(x - width/2, day1_means, width, yerr=day1_sems, 
                label='Day 1', alpha=0.8, capsize=5, color='lightsteelblue')
bars2 = ax4.bar(x + width/2, day5_means, width, yerr=day5_sems, 
                label='Day 5', alpha=0.8, capsize=5, color='lightcoral')

ax4.set_xlabel('Layer')
ax4.set_ylabel('Mean SMI ± SEM')
ax4.set_title('Comparison of Mean SMI Between Days')
ax4.set_xticks(x)
ax4.set_xticklabels(layers)
ax4.legend()
ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Add significance indicators between day 1 and day 5 for each layer
for i, result in enumerate(independent_results):
    if result['p_value'] < 0.05:
        if result['p_value'] < 0.001:
            sig_text = '***'
        elif result['p_value'] < 0.01:
            sig_text = '**'
        else:
            sig_text = '*'
        
        # Position the significance marker above the higher bar
        max_height = max(day1_means[i] + day1_sems[i], day5_means[i] + day5_sems[i])
        ax4.text(i, max_height + 0.05, sig_text, ha='center', va='bottom', fontweight='bold')

# Plot 5: Histogram of SMI distribution by layer (Day 5)
ax5 = fig.add_subplot(gs[1, 1])
for i, (data, layer, color) in enumerate(zip(day5_data, layers, colors)):
    ax5.hist(data, bins=15, alpha=0.6, color=color, label=layer, density=True)

ax5.axvline(0, color='k', linestyle='--', alpha=0.5)
ax5.set_title('Distribution of SMI by Layer (Day 5)')
ax5.set_xlabel('SMI Value')
ax5.set_ylabel('Density')
ax5.legend()

# Plot 6: Day 1 means with statistical comparison bars for comparison
ax6 = fig.add_subplot(gs[1, 2])

# Get Day 1 statistics for each layer
day1_data_list = [day1_l23, day1_l4, day1_l5, day1_l6]
day1_means = [np.mean(data) for data in day1_data_list]
day1_sems = [stats.sem(data) for data in day1_data_list]
layer_names_day1 = [f"{layer}\n(n={len(data)})" for layer, data in zip(layers, day1_data_list)]

# Create bar plot
bars = ax6.bar(range(len(day1_means)), day1_means, yerr=day1_sems, 
               capsize=10, color=colors, alpha=0.8)

ax6.set_xticks(range(len(day1_means)))
ax6.set_xticklabels(layer_names_day1)
ax6.axhline(0, color='k', linestyle='--', alpha=0.5)
ax6.set_title('Mean SMI by Layer (Day 1)')
ax6.set_ylabel('Mean SMI ± SEM')

# Add statistical comparison bars between layers (Day 1)
max_height_day1 = max(day1_means) + max(day1_sems) * 1.5
significant_comparisons_day1 = []

# Find significant comparisons between layers on Day 1
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day1_data_list[i], day1_data_list[j], equal_var=False)
        if p_value < 0.05:
            significant_comparisons_day1.append((i, j, p_value))

# Sort comparisons for logical ordering
significant_comparisons_day1.sort(key=lambda x: (x[0], x[1] - x[0]))

# Draw comparison bars for Day 1
bar_spacing_day1 = max_height_day1 * 0.1
current_height_day1 = max_height_day1

for bar_idx, (i, j, p_value) in enumerate(significant_comparisons_day1):
    bar_y = current_height_day1 + bar_spacing_day1 * bar_idx
    
    # Draw horizontal line
    ax6.plot([i, j], [bar_y, bar_y], 'k-', linewidth=1.5)
    
    # Draw vertical lines
    ax6.plot([i, i], [bar_y, bar_y-bar_spacing_day1*0.3], 'k-', linewidth=1.5)
    ax6.plot([j, j], [bar_y, bar_y-bar_spacing_day1*0.3], 'k-', linewidth=1.5)
    
    # Add significance marker
    if p_value < 0.001:
        sig_text = '***'
    elif p_value < 0.01:
        sig_text = '**'
    elif p_value < 0.05:
        sig_text = '*'
    
    ax6.text((i+j)/2, bar_y+bar_spacing_day1*0.1, sig_text, ha='center', va='bottom', fontsize=12, fontweight='bold')

# Adjust y-axis if we have comparisons
if significant_comparisons_day1:
    ax6.set_ylim(top=current_height_day1 + bar_spacing_day1 * (len(significant_comparisons_day1) + 1))

plt.tight_layout()
plt.show()

# Summary interpretation
print("\n=== SUMMARY INTERPRETATION ===\n")
print("Analysis of SMI differences between Day 1 and Day 5 populations:")
print("(Note: Using independent samples t-tests since cells are different between days)")
print()

significant_layers = [r['layer'] for r in independent_results if r['p_value'] < 0.05]
if significant_layers:
    print(f"Layers with significant differences: {', '.join(significant_layers)}")
else:
    print("No layers showed significant differences between days.")

print()
for result in independent_results:
    layer = result['layer']
    change = result['mean_change']
    p_val = result['p_value']
    
    if p_val < 0.05:
        direction = "higher" if change > 0 else "lower"
        magnitude = "large" if abs(result['effect_size']) > 0.8 else "medium" if abs(result['effect_size']) > 0.5 else "small"
        print(f"• {layer}: Day 5 SMI {direction} than Day 1 (p={p_val:.3f}, {magnitude} effect size)")
    else:
        print(f"• {layer}: No significant difference between days (p={p_val:.3f})")

print(f"\nRegarding your specific question about L2/3 trend:")
l23_change = [r for r in independent_results if r['layer'] == 'L2/3'][0]['mean_change']
l23_p = [r for r in independent_results if r['layer'] == 'L2/3'][0]['p_value']

if l23_change > 0:
    print(f"L2/3 shows higher mean SMI on Day 5 compared to Day 1 (difference: +{l23_change:.3f})")
    if l23_p < 0.05:
        print("This difference is statistically significant, suggesting an increasing trend in L2/3 spatial modulation.")
    else:
        print("However, this difference is not statistically significant.")
else:
    print(f"L2/3 shows lower mean SMI on Day 5 compared to Day 1 (difference: {l23_change:.3f})")

print("\nNote: Since these are different cell populations, we're comparing the")
print("overall spatial modulation properties between recording sessions rather")
print("than tracking changes in individual cells.")

print("\n=== LAYER COMPARISON SUMMARY ===\n")
print("Day 1 layer differences:")
day1_data_list = [day1_l23, day1_l4, day1_l5, day1_l6]
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day1_data_list[i], day1_data_list[j], equal_var=False)
        if p_value < 0.05:
            higher_layer = layers[j] if np.mean(day1_data_list[j]) > np.mean(day1_data_list[i]) else layers[i]
            lower_layer = layers[i] if np.mean(day1_data_list[j]) > np.mean(day1_data_list[i]) else layers[j]
            print(f"  {higher_layer} > {lower_layer} (p={p_value:.3f})")

print("\nDay 5 layer differences:")
day5_data_list = [day5_l23, day5_l4, day5_l5, day5_l6]
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day5_data_list[i], day5_data_list[j], equal_var=False)
        if p_value < 0.05:
            higher_layer = layers[j] if np.mean(day5_data_list[j]) > np.mean(day5_data_list[i]) else layers[i]
            lower_layer = layers[i] if np.mean(day5_data_list[j]) > np.mean(day5_data_list[i]) else layers[j]
            print(f"  {higher_layer} > {lower_layer} (p={p_value:.3f})")

# Create summary table
print("\n=== SUMMARY TABLE ===\n")
summary_data = []
for result in independent_results:
    day1_stats = [s for s in all_stats if s['layer'] == result['layer'] and s['day'] == 1][0]
    day5_stats = [s for s in all_stats if s['layer'] == result['layer'] and s['day'] == 5][0]
    
    summary_data.append({
        'Layer': result['layer'],
        'Day1_Mean': day1_stats['mean'],
        'Day1_SEM': day1_stats['sem'],
        'Day1_n': day1_stats['n'],
        'Day5_Mean': day5_stats['mean'],
        'Day5_SEM': day5_stats['sem'],
        'Day5_n': day5_stats['n'],
        'Difference': result['mean_change'],
        'p_value': result['p_value'],
        'Effect_Size': result['effect_size'],
        'Significant': 'Yes' if result['p_value'] < 0.05 else 'No'
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(3))

print("\n=== INTERPRETATION GUIDELINES ===")
print("Effect Size Interpretation (Cohen's d):")
print("  Small effect: 0.2-0.5")
print("  Medium effect: 0.5-0.8") 
print("  Large effect: >0.8")
print("\nStatistical significance: p < 0.05")
print("Multiple comparisons: Consider Bonferroni correction if needed (p < 0.0125 for 4 tests)")

print("\n=== ANALYSIS COMPLETE ===")
print("Script finished successfully. All statistical comparisons and visualizations generated.")

In [ ]:

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))
gs = GridSpec(1, 2, figure=fig)
fig.suptitle('SMI Comparison Across Layers Between Day 1 and Day 5', fontsize=24, fontweight='bold')

# Create dataframe for plotting
df = create_comparison_dataframe()
layers = ['L2/3', 'L4', 'L5', 'L6']
colors = ['lightblue', 'lightgreen', 'salmon', 'plum']

# Plot 1: Box plots by layer and day
ax1 = fig.add_subplot(gs[0, 0])
sns.boxplot(data=df, x='Layer', y='SMI', hue='Day', ax=ax1, palette=['lightsteelblue', 'lightcoral'])
ax1.set_title('SMI Distribution by Layer and Day')
ax1.set_ylabel('Spatial Modulation Index')
ax1.legend(title='Day')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)


# Add statistical comparison bars between layers (Day 5 only)
max_height = max(day5_means) + max(day5_sems) * 1.5
significant_comparisons = []

# Find significant comparisons between layers on Day 5
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day5_data[i], day5_data[j], equal_var=False)
        if p_value < 0.05:
            significant_comparisons.append((i, j, p_value))

# Sort comparisons for logical ordering
significant_comparisons.sort(key=lambda x: (x[0], x[1] - x[0]))

# Plot 3: Mean differences between days with statistical comparison
ax2 = fig.add_subplot(gs[0, 1])
mean_changes = [r['mean_change'] for r in independent_results]
p_values = [r['p_value'] for r in independent_results]

# Color bars based on significance
bar_colors = ['red' if p < 0.05 else 'gray' for p in p_values]
bars = ax2.bar(range(len(mean_changes)), mean_changes, color=bar_colors, alpha=0.7)
ax2.set_xticks(range(len(layers)))
ax2.set_xticklabels(layers)
ax2.set_title('Mean SMI Difference (Day 5 - Day 1)')
ax2.set_ylabel('Difference in Mean SMI')
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.8)
ax2.set_ylim(bottom=min(mean_changes) - 0.1, top=max(mean_changes) + 0.1)
# Add significance stars above bars
for i, (bar, p_val) in enumerate(zip(bars, p_values)):
    if p_val < 0.001:
        star = '***'
    elif p_val < 0.01:
        star = '**'
    elif p_val < 0.05:
        star = '*'
    else:
        star = 'ns'
    
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + (0.01 if height > 0 else -0.03),
             star, ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')

# # Plot 4: Side-by-side comparison of layer means
# ax4 = fig.add_subplot(gs[1, 0])

# # Prepare data for side-by-side bars
# x = np.arange(len(layers))
# width = 0.35

# day1_means = [np.mean(data) for data in [day1_l23, day1_l4, day1_l5, day1_l6]]
# day5_means = [np.mean(data) for data in [day5_l23, day5_l4, day5_l5, day5_l6]]
# day1_sems = [stats.sem(data) for data in [day1_l23, day1_l4, day1_l5, day1_l6]]
# day5_sems = [stats.sem(data) for data in [day5_l23, day5_l4, day5_l5, day5_l6]]

# bars1 = ax4.bar(x - width/2, day1_means, width, yerr=day1_sems, 
#                 label='Day 1', alpha=0.8, capsize=5, color='lightsteelblue')
# bars2 = ax4.bar(x + width/2, day5_means, width, yerr=day5_sems, 
#                 label='Day 5', alpha=0.8, capsize=5, color='lightcoral')

# ax4.set_xlabel('Layer')
# ax4.set_ylabel('Mean SMI ± SEM')
# ax4.set_title('Comparison of Mean SMI Between Days')
# ax4.set_xticks(x)
# ax4.set_xticklabels(layers)
# ax4.legend()
# ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# # Add significance indicators between day 1 and day 5 for each layer
# for i, result in enumerate(independent_results):
#     if result['p_value'] < 0.05:
#         if result['p_value'] < 0.001:
#             sig_text = '***'
#         elif result['p_value'] < 0.01:
#             sig_text = '**'
#         else:
#             sig_text = '*'
        
#         # Position the significance marker above the higher bar
#         max_height = max(day1_means[i] + day1_sems[i], day5_means[i] + day5_sems[i])
#         ax4.text(i, max_height + 0.05, sig_text, ha='center', va='bottom', fontweight='bold')

# # Plot 5: Histogram of SMI distribution by layer (Day 5)
# ax5 = fig.add_subplot(gs[1, 1])
# for i, (data, layer, color) in enumerate(zip(day5_data, layers, colors)):
#     ax5.hist(data, bins=15, alpha=0.6, color=color, label=layer, density=True)

# ax5.axvline(0, color='k', linestyle='--', alpha=0.5)
# ax5.set_title('Distribution of SMI by Layer (Day 5)')
# ax5.set_xlabel('SMI Value')
# ax5.set_ylabel('Density')
# ax5.legend()

# # Plot 6: Day 1 means with statistical comparison bars for comparison
# ax6 = fig.add_subplot(gs[1, 2])

# # Get Day 1 statistics for each layer
# day1_data_list = [day1_l23, day1_l4, day1_l5, day1_l6]
# day1_means = [np.mean(data) for data in day1_data_list]
# day1_sems = [stats.sem(data) for data in day1_data_list]
# layer_names_day1 = [f"{layer}\n(n={len(data)})" for layer, data in zip(layers, day1_data_list)]

# # Create bar plot
# bars = ax6.bar(range(len(day1_means)), day1_means, yerr=day1_sems, 
#                capsize=10, color=colors, alpha=0.8)

# ax6.set_xticks(range(len(day1_means)))
# ax6.set_xticklabels(layer_names_day1)
# ax6.axhline(0, color='k', linestyle='--', alpha=0.5)
# ax6.set_title('Mean SMI by Layer (Day 1)')
# ax6.set_ylabel('Mean SMI ± SEM')

# # Add statistical comparison bars between layers (Day 1)
# max_height_day1 = max(day1_means) + max(day1_sems) * 1.5
# significant_comparisons_day1 = []

# # Find significant comparisons between layers on Day 1
# for i in range(len(layers)):
#     for j in range(i+1, len(layers)):
#         t_stat, p_value = stats.ttest_ind(day1_data_list[i], day1_data_list[j], equal_var=False)
#         if p_value < 0.05:
#             significant_comparisons_day1.append((i, j, p_value))

# # Sort comparisons for logical ordering
# significant_comparisons_day1.sort(key=lambda x: (x[0], x[1] - x[0]))

# # Draw comparison bars for Day 1
# bar_spacing_day1 = max_height_day1 * 0.1
# current_height_day1 = max_height_day1

# for bar_idx, (i, j, p_value) in enumerate(significant_comparisons_day1):
#     bar_y = current_height_day1 + bar_spacing_day1 * bar_idx
    
#     # Draw horizontal line
#     ax6.plot([i, j], [bar_y, bar_y], 'k-', linewidth=1.5)
    
#     # Draw vertical lines
#     ax6.plot([i, i], [bar_y, bar_y-bar_spacing_day1*0.3], 'k-', linewidth=1.5)
#     ax6.plot([j, j], [bar_y, bar_y-bar_spacing_day1*0.3], 'k-', linewidth=1.5)
    
#     # Add significance marker
#     if p_value < 0.001:
#         sig_text = '***'
#     elif p_value < 0.01:
#         sig_text = '**'
#     elif p_value < 0.05:
#         sig_text = '*'
    
#     ax6.text((i+j)/2, bar_y+bar_spacing_day1*0.1, sig_text, ha='center', va='bottom', fontsize=12, fontweight='bold')

# # Adjust y-axis if we have comparisons
# if significant_comparisons_day1:
#     ax6.set_ylim(top=current_height_day1 + bar_spacing_day1 * (len(significant_comparisons_day1) + 1))

plt.tight_layout()
plt.show()

# Summary interpretation
print("\n=== SUMMARY INTERPRETATION ===\n")
print("Analysis of SMI differences between Day 1 and Day 5 populations:")
print("(Note: Using independent samples t-tests since cells are different between days)")
print()

significant_layers = [r['layer'] for r in independent_results if r['p_value'] < 0.05]
if significant_layers:
    print(f"Layers with significant differences: {', '.join(significant_layers)}")
else:
    print("No layers showed significant differences between days.")

print()
for result in independent_results:
    layer = result['layer']
    change = result['mean_change']
    p_val = result['p_value']
    
    if p_val < 0.05:
        direction = "higher" if change > 0 else "lower"
        magnitude = "large" if abs(result['effect_size']) > 0.8 else "medium" if abs(result['effect_size']) > 0.5 else "small"
        print(f"• {layer}: Day 5 SMI {direction} than Day 1 (p={p_val:.3f}, {magnitude} effect size)")
    else:
        print(f"• {layer}: No significant difference between days (p={p_val:.3f})")

print(f"\nRegarding your specific question about L2/3 trend:")
l23_change = [r for r in independent_results if r['layer'] == 'L2/3'][0]['mean_change']
l23_p = [r for r in independent_results if r['layer'] == 'L2/3'][0]['p_value']

if l23_change > 0:
    print(f"L2/3 shows higher mean SMI on Day 5 compared to Day 1 (difference: +{l23_change:.3f})")
    if l23_p < 0.05:
        print("This difference is statistically significant, suggesting an increasing trend in L2/3 spatial modulation.")
    else:
        print("However, this difference is not statistically significant.")
else:
    print(f"L2/3 shows lower mean SMI on Day 5 compared to Day 1 (difference: {l23_change:.3f})")

print("\nNote: Since these are different cell populations, we're comparing the")
print("overall spatial modulation properties between recording sessions rather")
print("than tracking changes in individual cells.")

print("\n=== LAYER COMPARISON SUMMARY ===\n")
print("Day 1 layer differences:")
day1_data_list = [day1_l23, day1_l4, day1_l5, day1_l6]
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day1_data_list[i], day1_data_list[j], equal_var=False)
        if p_value < 0.05:
            higher_layer = layers[j] if np.mean(day1_data_list[j]) > np.mean(day1_data_list[i]) else layers[i]
            lower_layer = layers[i] if np.mean(day1_data_list[j]) > np.mean(day1_data_list[i]) else layers[j]
            print(f"  {higher_layer} > {lower_layer} (p={p_value:.3f})")

print("\nDay 5 layer differences:")
day5_data_list = [day5_l23, day5_l4, day5_l5, day5_l6]
for i in range(len(layers)):
    for j in range(i+1, len(layers)):
        t_stat, p_value = stats.ttest_ind(day5_data_list[i], day5_data_list[j], equal_var=False)
        if p_value < 0.05:
            higher_layer = layers[j] if np.mean(day5_data_list[j]) > np.mean(day5_data_list[i]) else layers[i]
            lower_layer = layers[i] if np.mean(day5_data_list[j]) > np.mean(day5_data_list[i]) else layers[j]
            print(f"  {higher_layer} > {lower_layer} (p={p_value:.3f})")

# Create summary table
print("\n=== SUMMARY TABLE ===\n")
summary_data = []
for result in independent_results:
    day1_stats = [s for s in all_stats if s['layer'] == result['layer'] and s['day'] == 1][0]
    day5_stats = [s for s in all_stats if s['layer'] == result['layer'] and s['day'] == 5][0]
    
    summary_data.append({
        'Layer': result['layer'],
        'Day1_Mean': day1_stats['mean'],
        'Day1_SEM': day1_stats['sem'],
        'Day1_n': day1_stats['n'],
        'Day5_Mean': day5_stats['mean'],
        'Day5_SEM': day5_stats['sem'],
        'Day5_n': day5_stats['n'],
        'Difference': result['mean_change'],
        'p_value': result['p_value'],
        'Effect_Size': result['effect_size'],
        'Significant': 'Yes' if result['p_value'] < 0.05 else 'No'
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(3))

print("\n=== INTERPRETATION GUIDELINES ===")
print("Effect Size Interpretation (Cohen's d):")
print("  Small effect: 0.2-0.5")
print("  Medium effect: 0.5-0.8") 
print("  Large effect: >0.8")
print("\nStatistical significance: p < 0.05")
print("Multiple comparisons: Consider Bonferroni correction if needed (p < 0.0125 for 4 tests)")

print("\n=== ANALYSIS COMPLETE ===")
print("Script finished successfully. All statistical comparisons and visualizations generated.")